# Arthrodesis Procedures in Brazil's SUS (2015–2020)

**Comprehensive epidemiological analysis** of spinal arthrodesis (fusion) procedures in the Brazilian Universal Healthcare System.

**Pipeline:**
1. Data loading & cleaning
2. Descriptive statistics (demographics, diagnoses, costs)
3. Inflation adjustment (IPCA)
4. Population-adjusted procedure rates per 100,000
5. Age-sex standardization (WHO world standard)
6. Temporal trends (2015–2020)
7. Geographic visualizations (choropleth maps)
8. Specialist workforce analysis

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.figsize': (10, 6)
})

print('Libraries loaded successfully.')

In [ ]:
# Load the v7 dataset
df = pd.read_excel('data/artrodese_v7.xlsx', decimal=',')

# Convert date and filter to study period
df['dt_inter'] = pd.to_datetime(df['dt_inter'])
df = df[(df['dt_inter'] >= '2015-01-01') & (df['dt_inter'] <= '2020-12-31')].copy()

# Extract year and month for grouping
df['Year'] = df['dt_inter'].dt.year
df['Month'] = df['dt_inter'].dt.month
df['YearMonth'] = df['dt_inter'].dt.to_period('M')

print(f'Dataset: {df.shape[0]:,} procedures, {df.shape[1]} variables')
print(f'Period: {df["dt_inter"].min().strftime("%b %Y")} to {df["dt_inter"].max().strftime("%b %Y")}')
df.head(3)

In [ ]:
# Key columns overview
key_cols = [
    'Year', 'Age', 'Sex', 'def_race_color', 'Length_of_stay',
    'Total_value', 'Adj_VAL_TOTAL', 'US_Adj_VAL_TOT',
    'Main_diagnosis', 'def_diag_princ_subcat', 'Procedure_performed',
    'res_SIGLA_UF', 'int_SIGLA_UF', 'res_region', 'int_region',
    'death', 'UCI_use', 'age_groups'
]
print('Key variables:')
for col in key_cols:
    if col in df.columns:
        dtype = df[col].dtype
        n_miss = df[col].isna().sum()
        print(f'  {col:35s} {str(dtype):15s} missing={n_miss}')

## 2. Descriptive Statistics

In [ ]:
# --- Demographics ---
print('='*60)
print('DEMOGRAPHICS')
print('='*60)

# Age
print(f'\nAge: mean={df["Age"].mean():.1f}, median={df["Age"].median():.0f}, '
      f'SD={df["Age"].std():.1f}, range=[{df["Age"].min()}-{df["Age"].max()}]')

# Sex
print(f'\nSex distribution:')
sex_counts = df['Sex'].value_counts()
for sex, count in sex_counts.items():
    print(f'  {sex}: {count:,} ({count/len(df)*100:.1f}%)')

# Race
print(f'\nRace/Color:')
race_counts = df['def_race_color'].value_counts()
for race, count in race_counts.items():
    print(f'  {race}: {count:,} ({count/len(df)*100:.1f}%)')

# Age groups
print(f'\nAge groups:')
age_counts = df['age_groups'].value_counts().sort_index()
for ag, count in age_counts.items():
    print(f'  {ag}: {count:,} ({count/len(df)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age histogram
sns.histplot(df['Age'], bins=40, kde=True, color='steelblue', ax=axes[0])
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', label=f'Mean={df["Age"].mean():.1f}')
axes[0].axvline(df['Age'].median(), color='orange', linestyle='--', label=f'Median={df["Age"].median():.0f}')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Age by sex
sns.boxplot(data=df, x='Sex', y='Age', palette='Set2', ax=axes[1])
axes[1].set_title('Age by Sex')

plt.tight_layout()
plt.savefig('output/age_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# --- Top Diagnoses ---
print('='*60)
print('TOP 15 DIAGNOSES (ICD-10 subcategory)')
print('='*60)

diag_counts = df['def_diag_princ_subcat'].value_counts()
diag_pct = df['def_diag_princ_subcat'].value_counts(normalize=True) * 100
diag_table = pd.DataFrame({'N': diag_counts, '%': diag_pct}).head(15)
diag_table['Cumulative %'] = diag_table['%'].cumsum()
print(diag_table.to_string())

# Plot top 15
fig, ax = plt.subplots(figsize=(10, 6))
top15 = diag_table.reset_index()
top15.columns = ['Diagnosis', 'N', '%', 'Cumulative %']
sns.barplot(data=top15, x='N', y='Diagnosis', palette='viridis', ax=ax)
ax.set_title('Top 15 Primary Diagnoses')
ax.set_xlabel('Number of Procedures')
plt.tight_layout()
plt.savefig('output/top_diagnoses.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# --- Clinical Outcomes ---
print('='*60)
print('CLINICAL OUTCOMES')
print('='*60)

# Mortality
n_deaths = df['death'].sum()
print(f'\nIn-hospital mortality: {n_deaths:,} ({n_deaths/len(df)*100:.2f}%)')

# Length of stay
print(f'\nLength of stay (days):')
print(f'  Mean={df["Length_of_stay"].mean():.1f}, Median={df["Length_of_stay"].median():.0f}, '
      f'SD={df["Length_of_stay"].std():.1f}')
print(f'  IQR=[{df["Length_of_stay"].quantile(0.25):.0f}-{df["Length_of_stay"].quantile(0.75):.0f}]')

# ICU use
n_icu = df['UCI_use'].sum()
print(f'\nICU admission: {n_icu:,} ({n_icu/len(df)*100:.1f}%)')

# Costs
print(f'\nCost per procedure (BRL, inflation-adjusted):')
print(f'  Mean={df["Adj_VAL_TOTAL"].mean():,.2f}, Median={df["Adj_VAL_TOTAL"].median():,.2f}')
print(f'  SD={df["Adj_VAL_TOTAL"].std():,.2f}')
print(f'  Total cost: R$ {df["Adj_VAL_TOTAL"].sum():,.2f}')

print(f'\nCost per procedure (USD, inflation-adjusted):')
print(f'  Mean={df["US_Adj_VAL_TOT"].mean():,.2f}, Median={df["US_Adj_VAL_TOT"].median():,.2f}')

## 3. Inflation Adjustment Verification

The dataset includes pre-computed inflation-adjusted values (`Adj_VAL_TOTAL`) and USD conversions (`US_Adj_VAL_TOT`). Let's verify and visualize the adjustment.

In [ ]:
# IPCA 12-month inflation rates (% y-o-y) from the original analysis
ipca_data = {
    'DateTime': pd.to_datetime([
        '2015-01-01','2015-02-01','2015-03-01','2015-04-01','2015-05-01','2015-06-01',
        '2015-07-01','2015-08-01','2015-09-01','2015-10-01','2015-11-01','2015-12-01',
        '2016-01-01','2016-02-01','2016-03-01','2016-04-01','2016-05-01','2016-06-01',
        '2016-07-01','2016-08-01','2016-09-01','2016-10-01','2016-11-01','2016-12-01',
        '2017-01-01','2017-02-01','2017-03-01','2017-04-01','2017-05-01','2017-06-01',
        '2017-07-01','2017-08-01','2017-09-01','2017-10-01','2017-11-01','2017-12-01',
        '2018-01-01','2018-02-01','2018-03-01','2018-04-01','2018-05-01','2018-06-01',
        '2018-07-01','2018-08-01','2018-09-01','2018-10-01','2018-11-01','2018-12-01',
        '2019-01-01','2019-02-01','2019-03-01','2019-04-01','2019-05-01','2019-06-01',
        '2019-07-01','2019-08-01','2019-09-01','2019-10-01','2019-11-01','2019-12-01',
        '2020-01-01','2020-02-01','2020-03-01','2020-04-01','2020-05-01','2020-06-01',
        '2020-07-01','2020-08-01','2020-09-01','2020-10-01','2020-11-01','2020-12-01'
    ]),
    'Value': [
        7.14,7.70,8.13,8.17,8.47,8.89,9.56,9.53,9.49,9.93,10.48,10.67,
        10.71,10.36,9.39,9.28,9.32,8.84,8.74,8.97,8.48,7.87,6.99,6.29,
        5.35,4.76,4.57,4.08,3.60,3.00,2.71,2.46,2.54,2.70,2.80,2.95,
        2.86,2.84,2.68,2.76,2.86,4.39,4.48,4.19,4.53,4.56,4.05,3.75,
        3.78,3.89,4.58,4.94,4.66,3.37,3.22,3.43,2.89,2.54,3.27,4.31,
        4.19,4.01,3.30,2.40,1.88,2.13,2.31,2.44,3.14,3.92,4.31,4.52
    ]
}
ipca = pd.DataFrame(ipca_data)

# Compare nominal vs adjusted costs over time
cost_by_month = df.groupby('YearMonth').agg(
    nominal_mean=('Total_value', 'mean'),
    adjusted_mean=('Adj_VAL_TOTAL', 'mean'),
    count=('Total_value', 'size')
).reset_index()
cost_by_month['YearMonth'] = cost_by_month['YearMonth'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(cost_by_month['YearMonth'], cost_by_month['nominal_mean'], 'b-o', 
         markersize=3, label='Nominal (BRL)', alpha=0.7)
ax1.plot(cost_by_month['YearMonth'], cost_by_month['adjusted_mean'], 'r-o', 
         markersize=3, label='Inflation-adjusted (BRL)', alpha=0.7)
ax1.set_xlabel('Date')
ax1.set_ylabel('Mean Cost per Procedure (BRL)')
ax1.legend(loc='upper left')
ax1.set_title('Nominal vs Inflation-Adjusted Cost per Procedure')

ax2 = ax1.twinx()
ax2.fill_between(ipca['DateTime'], ipca['Value'], alpha=0.15, color='gray')
ax2.set_ylabel('IPCA 12-month (% y-o-y)', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

plt.tight_layout()
plt.savefig('output/inflation_adjustment.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Population-Adjusted Procedure Rates

In [ ]:
# IBGE population estimates (used in original analysis)
populacao_brasil = {
    'RO': 1777225, 'AC': 881935, 'AM': 4144597, 'RR': 605761,
    'PA': 8602865, 'AP': 845731, 'TO': 1572866, 'MA': 7075181,
    'PI': 3273227, 'CE': 9132078, 'RN': 3506853, 'PB': 4018127,
    'PE': 9557071, 'AL': 3337357, 'SE': 2298696, 'BA': 14873064,
    'MG': 21168791, 'ES': 4018650, 'RJ': 17264943, 'SP': 45919049,
    'PR': 11433957, 'SC': 7164788, 'RS': 11377239, 'MS': 2778986,
    'MT': 3484466, 'GO': 7018354, 'DF': 3015268
}

# Regions mapping
uf_to_region = {
    'RO': 'Norte', 'AC': 'Norte', 'AM': 'Norte', 'RR': 'Norte',
    'PA': 'Norte', 'AP': 'Norte', 'TO': 'Norte',
    'MA': 'Nordeste', 'PI': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'BA': 'Nordeste',
    'MG': 'Sudeste', 'ES': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste'
}

population = pd.DataFrame(list(populacao_brasil.items()), columns=['UF', 'Population'])
population['Region'] = population['UF'].map(uf_to_region)

# Rates by RESIDENCE state
df_res = df['res_SIGLA_UF'].value_counts().reset_index()
df_res.columns = ['UF', 'N']
df_res = df_res.merge(population, on='UF')
df_res['Rate_per_100k'] = df_res['N'] / df_res['Population'] * 100000
df_res = df_res.sort_values('Rate_per_100k', ascending=False)

# Rates by HOSPITAL state
df_int = df['int_SIGLA_UF'].value_counts().reset_index()
df_int.columns = ['UF', 'N']
df_int = df_int.merge(population, on='UF')
df_int['Rate_per_100k'] = df_int['N'] / df_int['Population'] * 100000
df_int = df_int.sort_values('Rate_per_100k', ascending=False)

print('TOP 10 STATES — Procedure Rate per 100,000 (by patient residence):')
print(df_res[['UF', 'Region', 'N', 'Population', 'Rate_per_100k']].head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# By residence
colors_res = [sns.color_palette('viridis', n_colors=len(df_res))[i] for i in range(len(df_res))]
sns.barplot(data=df_res, x='Rate_per_100k', y='UF', palette='viridis', ax=axes[0])
axes[0].set_title('By Patient Residence')
axes[0].set_xlabel('Procedures per 100,000 inhabitants')
axes[0].set_ylabel('State')

# By hospital
sns.barplot(data=df_int, x='Rate_per_100k', y='UF', palette='viridis', ax=axes[1])
axes[1].set_title('By Hospital Location')
axes[1].set_xlabel('Procedures per 100,000 inhabitants')
axes[1].set_ylabel('')

fig.suptitle('Arthrodesis Procedure Rates by State (2015–2020)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('output/state_rates_barplot.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Rates by REGION
region_pop = population.groupby('Region')['Population'].sum().reset_index()
region_n = df.groupby('res_region').size().reset_index(name='N')
region_n.columns = ['Region', 'N']

# Normalize region names if needed
region_rates = region_n.merge(region_pop, on='Region', how='left')
region_rates['Rate_per_100k'] = region_rates['N'] / region_rates['Population'] * 100000
region_rates = region_rates.sort_values('Rate_per_100k', ascending=False)

print('PROCEDURE RATES BY REGION:')
print(region_rates.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=region_rates, x='Rate_per_100k', y='Region', palette='coolwarm', ax=ax)
ax.set_title('Arthrodesis Rate by Region (per 100,000)')
ax.set_xlabel('Procedures per 100,000 inhabitants')
plt.tight_layout()
plt.savefig('output/region_rates.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Age-Sex Standardization (WHO World Standard Population)

In [ ]:
# WHO World Standard Population weights (Segi, modified by Doll)
who_standard = {
    '0-4': 8860, '5-9': 8690, '10-14': 8600, '15-19': 8470,
    '20-24': 8220, '25-29': 7930, '30-34': 7610, '35-39': 7150,
    '40-44': 6590, '45-49': 6040, '50-54': 5370, '55-59': 4550,
    '60-64': 3720, '65-69': 2960, '70-74': 2210, '75-79': 1520,
    '80-84': 910, '85+': 630
}
total_std = sum(who_standard.values())

# Create age bins matching WHO groups
age_bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 200]
age_labels = list(who_standard.keys())
df['age_group_who'] = pd.cut(df['Age'], bins=age_bins, right=False, labels=age_labels)

def compute_age_standardized_rate(subset_df, pop_total, standard=who_standard, total_std=total_std):
    """Compute direct age-standardized rate per 100,000."""
    # This is a simplified approach: assumes uniform age distribution in the population
    # For a proper standardization, we'd need age-specific population denominators per state
    age_counts = subset_df['age_group_who'].value_counts()
    weighted_sum = 0
    for ag, weight in standard.items():
        n_cases = age_counts.get(ag, 0)
        # Age-specific rate * standard weight
        weighted_sum += (n_cases / pop_total * 100000) * (weight / total_std)
    return weighted_sum

# Age-specific procedure counts (national)
age_dist = df['age_group_who'].value_counts().sort_index()
age_dist_pct = (age_dist / age_dist.sum() * 100).round(1)

print('AGE DISTRIBUTION OF PROCEDURES (WHO groups):')
age_table = pd.DataFrame({'N': age_dist, '%': age_dist_pct})
print(age_table.to_string())

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
age_table['N'].plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Procedure Count by WHO Age Group')
ax.set_xlabel('Age Group')
ax.set_ylabel('Number of Procedures')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('output/age_who_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Temporal Trends (2015–2020)

In [ ]:
# Yearly procedure counts
yearly = df.groupby('Year').agg(
    N=('Year', 'size'),
    mean_age=('Age', 'mean'),
    mortality_pct=('death', lambda x: x.sum()/len(x)*100),
    mean_los=('Length_of_stay', 'mean'),
    mean_cost_adj=('Adj_VAL_TOTAL', 'mean'),
    total_cost_adj=('Adj_VAL_TOTAL', 'sum'),
    icu_pct=('UCI_use', lambda x: x.sum()/len(x)*100)
).reset_index()

print('YEARLY SUMMARY:')
print(yearly.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Volume
axes[0, 0].plot(yearly['Year'], yearly['N'], 'b-o', linewidth=2)
axes[0, 0].set_title('Procedure Volume')
axes[0, 0].set_ylabel('N')
axes[0, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Mean age
axes[0, 1].plot(yearly['Year'], yearly['mean_age'], 'g-o', linewidth=2)
axes[0, 1].set_title('Mean Age')
axes[0, 1].set_ylabel('Years')

# Mortality
axes[0, 2].plot(yearly['Year'], yearly['mortality_pct'], 'r-o', linewidth=2)
axes[0, 2].set_title('In-Hospital Mortality')
axes[0, 2].set_ylabel('%')

# LOS
axes[1, 0].plot(yearly['Year'], yearly['mean_los'], 'm-o', linewidth=2)
axes[1, 0].set_title('Mean Length of Stay')
axes[1, 0].set_ylabel('Days')

# Mean cost
axes[1, 1].plot(yearly['Year'], yearly['mean_cost_adj'], 'orange', marker='o', linewidth=2)
axes[1, 1].set_title('Mean Cost (Adjusted BRL)')
axes[1, 1].set_ylabel('BRL')
axes[1, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# ICU
axes[1, 2].plot(yearly['Year'], yearly['icu_pct'], 'c-o', linewidth=2)
axes[1, 2].set_title('ICU Admission')
axes[1, 2].set_ylabel('%')

for ax in axes.flat:
    ax.set_xlabel('Year')
    ax.set_xticks(yearly['Year'])

fig.suptitle('Temporal Trends in Arthrodesis Procedures (2015–2020)', fontsize=14)
plt.tight_layout()
plt.savefig('output/temporal_trends.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly time series
monthly = df.groupby('YearMonth').size().reset_index(name='N')
monthly['YearMonth'] = monthly['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['YearMonth'], monthly['N'], 'steelblue', linewidth=1.5)
ax.fill_between(monthly['YearMonth'], monthly['N'], alpha=0.2, color='steelblue')

# Mark COVID-19 onset
covid_start = pd.Timestamp('2020-03-01')
ax.axvline(covid_start, color='red', linestyle='--', alpha=0.7, label='COVID-19 onset (Mar 2020)')

ax.set_title('Monthly Arthrodesis Volume (2015–2020)')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Procedures')
ax.legend()
plt.tight_layout()
plt.savefig('output/monthly_trends.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Trends by region
region_yearly = df.groupby(['Year', 'res_region']).size().reset_index(name='N')

fig, ax = plt.subplots(figsize=(12, 6))
for region in sorted(region_yearly['res_region'].unique()):
    subset = region_yearly[region_yearly['res_region'] == region]
    ax.plot(subset['Year'], subset['N'], '-o', linewidth=2, markersize=5, label=region)

ax.set_title('Yearly Procedure Volume by Region')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Procedures')
ax.legend(title='Region')
ax.set_xticks(range(2015, 2021))
plt.tight_layout()
plt.savefig('output/region_trends.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Geographic Visualizations (Choropleth Maps)

In [ ]:
import geobr

# Load Brazilian state boundaries
states_gdf = geobr.read_state(year=2019)
print(f'Loaded {len(states_gdf)} states')
print(f'Columns: {list(states_gdf.columns)}')
states_gdf.head(3)

In [ ]:
# Merge geographic data with procedure rates
states_res = states_gdf.merge(df_res, how='left', left_on='abbrev_state', right_on='UF')
states_int = states_gdf.merge(df_int, how='left', left_on='abbrev_state', right_on='UF')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# By residence
states_res.plot(
    column='Rate_per_100k', cmap='OrRd', legend=True, ax=axes[0],
    edgecolor='gray', linewidth=0.5,
    legend_kwds={'label': 'Rate per 100,000', 'orientation': 'horizontal', 'shrink': 0.7}
)
axes[0].set_title('By Patient Residence', fontsize=12)
axes[0].axis('off')

# By hospital
states_int.plot(
    column='Rate_per_100k', cmap='OrRd', legend=True, ax=axes[1],
    edgecolor='gray', linewidth=0.5,
    legend_kwds={'label': 'Rate per 100,000', 'orientation': 'horizontal', 'shrink': 0.7}
)
axes[1].set_title('By Hospital Location', fontsize=12)
axes[1].axis('off')

fig.suptitle('Arthrodesis Procedure Rates in SUS (2015–2020)', fontsize=14)
plt.tight_layout()
plt.savefig('output/choropleth_rates.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Quartile maps
states_res['quartile'] = pd.qcut(states_res['Rate_per_100k'], 4, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])
states_int['quartile'] = pd.qcut(states_int['Rate_per_100k'], 4, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

states_res.plot(column='quartile', cmap='RdYlGn_r', legend=True, ax=axes[0],
                edgecolor='gray', linewidth=0.5, categorical=True)
axes[0].set_title('Rate Quartiles (Residence)', fontsize=12)
axes[0].axis('off')

states_int.plot(column='quartile', cmap='RdYlGn_r', legend=True, ax=axes[1],
                edgecolor='gray', linewidth=0.5, categorical=True)
axes[1].set_title('Rate Quartiles (Hospital)', fontsize=12)
axes[1].axis('off')

fig.suptitle('Procedure Rate Quartiles by State', fontsize=14)
plt.tight_layout()
plt.savefig('output/choropleth_quartiles.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Mortality rate map
mortality_by_state = df.groupby('res_SIGLA_UF').agg(
    N=('death', 'size'),
    deaths=('death', 'sum')
).reset_index()
mortality_by_state['mortality_pct'] = mortality_by_state['deaths'] / mortality_by_state['N'] * 100

states_mort = states_gdf.merge(mortality_by_state, how='left', left_on='abbrev_state', right_on='res_SIGLA_UF')

fig, ax = plt.subplots(figsize=(8, 8))
states_mort.plot(
    column='mortality_pct', cmap='Reds', legend=True, ax=ax,
    edgecolor='gray', linewidth=0.5,
    legend_kwds={'label': 'Mortality (%)', 'orientation': 'horizontal', 'shrink': 0.7}
)
ax.set_title('In-Hospital Mortality Rate by State', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig('output/choropleth_mortality.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. Specialist Workforce Analysis

In [ ]:
# Neurosurgeons per state
neurosurgeons = {
    'AC': 17, 'AL': 42, 'AP': 7, 'AM': 36, 'BA': 119, 'CE': 75,
    'DF': 112, 'ES': 111, 'GO': 105, 'MA': 60, 'MT': 55, 'MS': 49,
    'MG': 393, 'PA': 64, 'PB': 49, 'PR': 218, 'PE': 101, 'PI': 42,
    'RJ': 388, 'RN': 49, 'RS': 263, 'RO': 37, 'RR': 6, 'SC': 113,
    'SP': 1117, 'SE': 29, 'TO': 25
}

# Orthopedists per state
orthopedists = {
    'AC': 34, 'AL': 128, 'AP': 37, 'AM': 164, 'BA': 767, 'CE': 419,
    'DF': 606, 'ES': 433, 'GO': 623, 'MA': 255, 'MT': 284, 'MS': 279,
    'MG': 1941, 'PA': 225, 'PB': 226, 'PR': 1168, 'PE': 584, 'PI': 170,
    'RJ': 1792, 'RN': 179, 'RS': 1139, 'RO': 128, 'RR': 30, 'SC': 738,
    'SP': 5313, 'SE': 132, 'TO': 112
}

# Build specialist dataframe
df_spec = population.copy()
df_spec['Neurosurgeons'] = df_spec['UF'].map(neurosurgeons)
df_spec['Orthopedists'] = df_spec['UF'].map(orthopedists)
df_spec['Neuro_per_100k'] = df_spec['Neurosurgeons'] / df_spec['Population'] * 100000
df_spec['Ortho_per_100k'] = df_spec['Orthopedists'] / df_spec['Population'] * 100000
df_spec['Spine_specialists'] = df_spec['Neurosurgeons'] + df_spec['Orthopedists']
df_spec['Spine_per_100k'] = df_spec['Spine_specialists'] / df_spec['Population'] * 100000

# Merge with procedure rates
df_spec = df_spec.merge(df_res[['UF', 'Rate_per_100k']], on='UF')
df_spec.rename(columns={'Rate_per_100k': 'Procedure_rate'}, inplace=True)

# Correlation
r_neuro, p_neuro = stats.pearsonr(df_spec['Neuro_per_100k'], df_spec['Procedure_rate'])
r_ortho, p_ortho = stats.pearsonr(df_spec['Ortho_per_100k'], df_spec['Procedure_rate'])
r_spine, p_spine = stats.pearsonr(df_spec['Spine_per_100k'], df_spec['Procedure_rate'])

print(f'Correlation: Neurosurgeons/100k vs Procedure rate: r={r_neuro:.3f}, p={p_neuro:.4f}')
print(f'Correlation: Orthopedists/100k vs Procedure rate:  r={r_ortho:.3f}, p={p_ortho:.4f}')
print(f'Correlation: All spine/100k vs Procedure rate:     r={r_spine:.3f}, p={p_spine:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Neurosurgeons
axes[0].scatter(df_spec['Neuro_per_100k'], df_spec['Procedure_rate'], c='steelblue', s=50, alpha=0.7)
for _, row in df_spec.iterrows():
    axes[0].annotate(row['UF'], (row['Neuro_per_100k'], row['Procedure_rate']), fontsize=7, alpha=0.8)
z = np.polyfit(df_spec['Neuro_per_100k'], df_spec['Procedure_rate'], 1)
p_line = np.poly1d(z)
x_range = np.linspace(df_spec['Neuro_per_100k'].min(), df_spec['Neuro_per_100k'].max(), 50)
axes[0].plot(x_range, p_line(x_range), 'r--', alpha=0.5)
axes[0].set_title(f'Neurosurgeons (r={r_neuro:.2f}, p={p_neuro:.3f})')
axes[0].set_xlabel('Neurosurgeons per 100,000')
axes[0].set_ylabel('Procedure Rate per 100,000')

# Orthopedists
axes[1].scatter(df_spec['Ortho_per_100k'], df_spec['Procedure_rate'], c='green', s=50, alpha=0.7)
for _, row in df_spec.iterrows():
    axes[1].annotate(row['UF'], (row['Ortho_per_100k'], row['Procedure_rate']), fontsize=7, alpha=0.8)
z = np.polyfit(df_spec['Ortho_per_100k'], df_spec['Procedure_rate'], 1)
p_line = np.poly1d(z)
x_range = np.linspace(df_spec['Ortho_per_100k'].min(), df_spec['Ortho_per_100k'].max(), 50)
axes[1].plot(x_range, p_line(x_range), 'r--', alpha=0.5)
axes[1].set_title(f'Orthopedists (r={r_ortho:.2f}, p={p_ortho:.3f})')
axes[1].set_xlabel('Orthopedists per 100,000')
axes[1].set_ylabel('Procedure Rate per 100,000')

# Combined
axes[2].scatter(df_spec['Spine_per_100k'], df_spec['Procedure_rate'], c='purple', s=50, alpha=0.7)
for _, row in df_spec.iterrows():
    axes[2].annotate(row['UF'], (row['Spine_per_100k'], row['Procedure_rate']), fontsize=7, alpha=0.8)
z = np.polyfit(df_spec['Spine_per_100k'], df_spec['Procedure_rate'], 1)
p_line = np.poly1d(z)
x_range = np.linspace(df_spec['Spine_per_100k'].min(), df_spec['Spine_per_100k'].max(), 50)
axes[2].plot(x_range, p_line(x_range), 'r--', alpha=0.5)
axes[2].set_title(f'Combined Spine Specialists (r={r_spine:.2f}, p={p_spine:.3f})')
axes[2].set_xlabel('Specialists per 100,000')
axes[2].set_ylabel('Procedure Rate per 100,000')

fig.suptitle('Specialist Density vs Arthrodesis Procedure Rate', fontsize=14)
plt.tight_layout()
plt.savefig('output/specialist_correlation.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. Forest Plot — Regional Rate Comparison

In [ ]:
# Compute yearly rates per state with 95% CI (Poisson-based)
# For rare events, CI = rate ± 1.96 * sqrt(rate / population) * 100000
n_years = 6  # 2015-2020

forest_data = df_res.copy()
forest_data['annual_rate'] = forest_data['Rate_per_100k'] / n_years
forest_data['se'] = np.sqrt(forest_data['N'] / (forest_data['Population'] ** 2)) * 100000 / n_years
forest_data['ci_low'] = forest_data['annual_rate'] - 1.96 * forest_data['se']
forest_data['ci_high'] = forest_data['annual_rate'] + 1.96 * forest_data['se']
forest_data = forest_data.sort_values('annual_rate', ascending=True)

# National average
national_rate = forest_data['N'].sum() / forest_data['Population'].sum() * 100000 / n_years

fig, ax = plt.subplots(figsize=(10, 12))

y_pos = range(len(forest_data))
ax.errorbar(
    forest_data['annual_rate'], y_pos,
    xerr=[forest_data['annual_rate'] - forest_data['ci_low'],
          forest_data['ci_high'] - forest_data['annual_rate']],
    fmt='D', color='steelblue', ecolor='gray', elinewidth=1, capsize=3, markersize=5
)
ax.axvline(national_rate, color='red', linestyle='--', alpha=0.6, label=f'National avg: {national_rate:.1f}')

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row['UF']} ({row['Region']})" for _, row in forest_data.iterrows()])
ax.set_xlabel('Annual Procedure Rate per 100,000 (95% CI)')
ax.set_title('Forest Plot — Annual Arthrodesis Rate by State')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('output/forest_plot_state_rates.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Summary Tables for Export

In [ ]:
# Table 1: State-level summary
table1 = df_res[['UF', 'Region', 'N', 'Population', 'Rate_per_100k']].copy()
table1.columns = ['State', 'Region', 'Procedures (N)', 'Population', 'Rate per 100,000']
table1['Rate per 100,000'] = table1['Rate per 100,000'].round(1)

# Add mortality and LOS per state
state_outcomes = df.groupby('res_SIGLA_UF').agg(
    mortality=('death', 'mean'),
    mean_los=('Length_of_stay', 'mean'),
    mean_cost=('Adj_VAL_TOTAL', 'mean')
).reset_index()
state_outcomes.columns = ['State', 'Mortality (%)', 'Mean LOS (days)', 'Mean Cost (BRL)']
state_outcomes['Mortality (%)'] = (state_outcomes['Mortality (%)'] * 100).round(2)
state_outcomes['Mean LOS (days)'] = state_outcomes['Mean LOS (days)'].round(1)
state_outcomes['Mean Cost (BRL)'] = state_outcomes['Mean Cost (BRL)'].round(2)

table1 = table1.merge(state_outcomes, on='State')

# Save
table1.to_csv('output/table1_state_summary.csv', index=False)
print('Table 1 — State Summary (sorted by rate):')
print(table1.to_string(index=False))

In [ ]:
# Table 2: Yearly summary
yearly.columns = ['Year', 'N', 'Mean Age', 'Mortality (%)', 'Mean LOS', 'Mean Cost (adj BRL)', 'Total Cost (adj BRL)', 'ICU (%)']
yearly['Mortality (%)'] = yearly['Mortality (%)'].round(2)
yearly['Mean LOS'] = yearly['Mean LOS'].round(1)
yearly['Mean Age'] = yearly['Mean Age'].round(1)
yearly['Mean Cost (adj BRL)'] = yearly['Mean Cost (adj BRL)'].round(2)
yearly['Total Cost (adj BRL)'] = yearly['Total Cost (adj BRL)'].round(2)
yearly['ICU (%)'] = yearly['ICU (%)'].round(1)

yearly.to_csv('output/table2_yearly_summary.csv', index=False)
print('Table 2 — Yearly Summary:')
print(yearly.to_string(index=False))

In [ ]:
print('='*60)
print('ANALYSIS COMPLETE')
print('='*60)
print(f'\nTotal procedures analyzed: {len(df):,}')
print(f'Period: 2015–2020 ({n_years} years)')
print(f'States: {df["res_SIGLA_UF"].nunique()}')
print(f'\nOutputs saved to output/:')

import os
for f in sorted(os.listdir('output')):
    size = os.path.getsize(f'output/{f}')
    print(f'  {f} ({size/1024:.0f} KB)')

## 11. Weinstein Variation Analysis

Replicating the methodology from **Weinstein et al. (2004)** *"United States' trends and regional variations in lumbar spine surgery: 1992–2003"* (Spine, 2006) and the Dartmouth Atlas of Health Care.

**Key metrics:**
- **Standardized Discharge Ratio (SDR)** = Observed / Expected rate (1.0 = national average)
- **Coefficient of Variation (CV)** = SD / Mean × 100
- **Extremal Ratio** = max(rate) / min(rate)
- **Interquartile Ratio (IQR ratio)** = Q3 / Q1

In [ ]:
# ── 11a. Compute Weinstein variation metrics (state-level) ──

# Annual rate per state (already computed in forest_data from Section 9)
# Recompute cleanly here for clarity
n_years = 6
state_rates = df.groupby('res_SIGLA_UF').size().reset_index(name='N')
state_rates = state_rates.merge(
    pd.DataFrame(list(populacao_brasil.items()), columns=['res_SIGLA_UF', 'Population']),
    on='res_SIGLA_UF'
)
state_rates['annual_rate'] = state_rates['N'] / state_rates['Population'] * 100000 / n_years

# National average rate (expected)
national_rate = state_rates['N'].sum() / state_rates['Population'].sum() * 100000 / n_years

# Standardized Discharge Ratio
state_rates['SDR'] = state_rates['annual_rate'] / national_rate

# ── Variation metrics ──
rates = state_rates['annual_rate']
sdrs = state_rates['SDR']

cv = rates.std() / rates.mean() * 100
extremal_ratio = rates.max() / rates.min()
q1, q3 = rates.quantile(0.25), rates.quantile(0.75)
iqr_ratio = q3 / q1
mean_sdr = sdrs.mean()

# Systematic component of variation (SCV) — McPherson et al.
# SCV = (Var(observed/expected) - mean(1/expected)) × 100
# Approximation for Poisson: SCV ≈ (Var(SDR) - 1/mean_N) * 100
mean_n = state_rates['N'].mean()
scv = (sdrs.var() - 1 / mean_n) * 100

print('='*60)
print('WEINSTEIN VARIATION METRICS — All Arthrodesis (State-level)')
print('='*60)
print(f'  Number of areas (states):       {len(state_rates)}')
print(f'  National annual rate/100k:      {national_rate:.2f}')
print(f'  Mean state annual rate/100k:    {rates.mean():.2f}')
print(f'  SD of state rates:              {rates.std():.2f}')
print(f'  ─────────────────────────────────')
print(f'  Coefficient of Variation (CV):  {cv:.1f}%')
print(f'  Extremal Ratio (max/min):       {extremal_ratio:.2f}')
print(f'  IQR Ratio (Q3/Q1):             {iqr_ratio:.2f}')
print(f'  Mean SDR:                       {mean_sdr:.3f}')
print(f'  Systematic Component (SCV):     {scv:.2f}')
print(f'  ─────────────────────────────────')
print(f'  Range: {rates.min():.2f} – {rates.max():.2f} per 100k/year')
print(f'  IQR:   {q1:.2f} – {q3:.2f} per 100k/year')

# Compare with Weinstein's US data for lumbar fusion
print(f'\n  ── Comparison with Weinstein (US Lumbar Fusion) ──')
print(f'  {"Metric":<30} {"Brazil (SUS)":<15} {"US (Weinstein)"}')
print(f'  {"CV (%)":<30} {cv:<15.1f} {"49.5"}')
print(f'  {"Extremal Ratio":<30} {extremal_ratio:<15.2f} {"21.0"}')
print(f'  {"IQR Ratio":<30} {iqr_ratio:<15.2f} {"2.01"}')

In [ ]:
# ── 11b. Standardized Discharge Ratio — Beeswarm Plot (Weinstein-style) ──

fig, ax = plt.subplots(figsize=(8, 10))

# Sort by SDR for the strip/beeswarm effect
state_rates_sorted = state_rates.sort_values('SDR')

# Use a strip plot with jitter to mimic Weinstein's beeswarm
# x=0 (single category), y=SDR on log scale
np.random.seed(42)
jitter = np.random.normal(0, 0.06, len(state_rates_sorted))

ax.scatter(
    jitter, state_rates_sorted['SDR'],
    s=60, c='#1f4e79', alpha=0.8, edgecolors='white', linewidths=0.5, zorder=3
)

# Annotate each point with state abbreviation
for _, row in state_rates_sorted.iterrows():
    x_offset = 0.15
    ax.annotate(
        row['res_SIGLA_UF'],
        (jitter[state_rates_sorted.index.get_loc(row.name)], row['SDR']),
        fontsize=7, alpha=0.7, ha='left',
        xytext=(x_offset, 0), textcoords='offset points'
    )

# Reference line at SDR = 1.0
ax.axhline(1.0, color='red', linestyle='-', linewidth=1.5, alpha=0.6, label='National average (SDR=1.0)')

# Log scale y-axis (matching Weinstein)
ax.set_yscale('log')
ax.set_ylim(0.03, 20)
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_minor_formatter(mticker.NullFormatter())
ax.set_yticks([0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0])
ax.get_yaxis().set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}' if x >= 1 else f'{x:.2f}'))

# Style
ax.set_xlim(-0.5, 0.5)
ax.set_xticks([0])
ax.set_xticklabels(['Spinal\nArthrodesis'])
ax.set_ylabel('Standardized Discharge Ratio (log scale)', fontsize=12)
ax.set_title('Geographic Variation in Spinal Arthrodesis\n(Brazilian States, 2015–2020)', fontsize=13)
ax.legend(loc='upper right', fontsize=9)

# Add metrics text box
textstr = (f'Mean SDR: {mean_sdr:.2f}\n'
           f'Extremal ratio: {extremal_ratio:.1f}\n'
           f'IQR ratio: {iqr_ratio:.2f}\n'
           f'CV: {cv:.1f}%')
props = dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
ax.text(0.02, 0.02, textstr, transform=ax.transAxes, fontsize=9,
        verticalalignment='bottom', bbox=props)

# Light yellow background like Weinstein
ax.set_facecolor('#fffff0')
fig.patch.set_facecolor('white')

ax.grid(axis='y', alpha=0.3)
ax.grid(axis='x', visible=False)

plt.tight_layout()
plt.savefig('output/weinstein_sdr_beeswarm.png', dpi=200, bbox_inches='tight')
plt.show()

## 12. Diagnosis-Specific Variation Analysis

Following Weinstein's approach of separating procedures by indication — we classify arthrodesis by spinal region using ICD-10 primary diagnosis:

| Category | ICD-10 Codes | Description |
|----------|-------------|-------------|
| **Lumbar degenerative** | M51.x, M47.8, M47.1, M48.0 | Disc herniation, spondylosis, spinal stenosis |
| **Cervical degenerative** | M50.x, M47.2 | Cervical disc disorders, cervical spondylosis |
| **Trauma (fractures)** | S12.x, S22.x, S32.x, T91.1 | Cervical/thoracic/lumbar fractures |
| **Deformity** | M40.x, M41.x, M43.1 | Kyphosis, scoliosis, spondylolisthesis |
| **Other** | All remaining | Neoplasm, infection, other |

In [ ]:
# ── 12a. Classify diagnoses by spinal region/indication ──

def classify_diagnosis(icd):
    """Classify ICD-10 primary diagnosis into spinal region categories."""
    if pd.isna(icd):
        return 'Other'
    icd = str(icd).strip().upper()
    
    # Lumbar degenerative
    if icd.startswith('M51'):      # Lumbar disc disorders
        return 'Lumbar degenerative'
    if icd in ('M47.8', 'M47.81', 'M47.89', 'M47.1', 'M47.10', 'M47.11', 'M47.12',
               'M47.13', 'M47.14', 'M47.15', 'M47.16', 'M47.17', 'M47.19'):
        return 'Lumbar degenerative'
    if icd.startswith('M48.0'):    # Spinal stenosis (lumbar most common)
        return 'Lumbar degenerative'
    
    # Cervical degenerative
    if icd.startswith('M50'):      # Cervical disc disorders
        return 'Cervical degenerative'
    if icd.startswith('M47.2'):    # Cervical spondylosis with myelopathy
        return 'Cervical degenerative'
    
    # Trauma / Fractures
    if icd.startswith('S12'):      # Cervical fractures
        return 'Trauma'
    if icd.startswith('S22'):      # Thoracic fractures
        return 'Trauma'
    if icd.startswith('S32'):      # Lumbar fractures
        return 'Trauma'
    if icd.startswith('S13'):      # Cervical dislocation
        return 'Trauma'
    if icd.startswith('S14'):      # Cervical cord injury
        return 'Trauma'
    if icd.startswith('S24'):      # Thoracic cord injury
        return 'Trauma'
    if icd.startswith('S34'):      # Lumbar cord injury
        return 'Trauma'
    if icd.startswith('T91'):      # Sequelae of spine/trunk injuries
        return 'Trauma'
    
    # Deformity
    if icd.startswith('M40'):      # Kyphosis
        return 'Deformity'
    if icd.startswith('M41'):      # Scoliosis
        return 'Deformity'
    if icd.startswith('M43.1'):    # Spondylolisthesis
        return 'Deformity'
    
    # Other spondylopathies that could be lumbar
    if icd.startswith('M47'):      # Remaining spondylosis (unspecified)
        return 'Lumbar degenerative'
    if icd.startswith('M48'):      # Remaining spinal stenosis
        return 'Lumbar degenerative'
    if icd.startswith('M43'):      # Remaining deforming dorsopathies
        return 'Deformity'
    
    return 'Other'

df['diag_category'] = df['Main_diagnosis'].apply(classify_diagnosis)

# Summary
cat_counts = df['diag_category'].value_counts()
cat_pct = (cat_counts / len(df) * 100).round(1)

print('DIAGNOSIS CLASSIFICATION:')
print(f'{"Category":<25} {"N":>8} {"(%)":>8}')
print('-'*43)
for cat in ['Lumbar degenerative', 'Cervical degenerative', 'Trauma', 'Deformity', 'Other']:
    n = cat_counts.get(cat, 0)
    p = cat_pct.get(cat, 0)
    print(f'{cat:<25} {n:>8,} {p:>7.1f}%')
print('-'*43)
print(f'{"TOTAL":<25} {len(df):>8,} {"100.0":>7}%')

In [ ]:
# ── 12b. Per-category variation metrics and state-level rates ──

categories = ['Lumbar degenerative', 'Cervical degenerative', 'Trauma', 'Deformity']

pop_df = pd.DataFrame(list(populacao_brasil.items()), columns=['UF', 'Population'])
variation_results = {}

for cat in categories:
    cat_df = df[df['diag_category'] == cat]
    
    # State-level counts
    cat_state = cat_df.groupby('res_SIGLA_UF').size().reset_index(name='N')
    cat_state = cat_state.merge(pop_df, left_on='res_SIGLA_UF', right_on='UF')
    cat_state['annual_rate'] = cat_state['N'] / cat_state['Population'] * 100000 / n_years
    
    # National expected rate
    cat_national = cat_state['N'].sum() / cat_state['Population'].sum() * 100000 / n_years
    cat_state['SDR'] = cat_state['annual_rate'] / cat_national
    
    # Variation metrics
    r = cat_state['annual_rate']
    s = cat_state['SDR']
    
    # Handle states with 0 procedures (avoid division by zero in extremal ratio)
    r_nonzero = r[r > 0]
    
    variation_results[cat] = {
        'N_total': len(cat_df),
        'N_states': len(cat_state),
        'National_rate': cat_national,
        'Mean_rate': r.mean(),
        'SD_rate': r.std(),
        'CV': r.std() / r.mean() * 100 if r.mean() > 0 else np.nan,
        'Extremal_ratio': r_nonzero.max() / r_nonzero.min() if len(r_nonzero) > 1 else np.nan,
        'Q1': r.quantile(0.25),
        'Q3': r.quantile(0.75),
        'IQR_ratio': r.quantile(0.75) / r.quantile(0.25) if r.quantile(0.25) > 0 else np.nan,
        'Mean_SDR': s.mean(),
        'state_data': cat_state
    }

# Print comparison table
print('='*80)
print('WEINSTEIN VARIATION METRICS BY DIAGNOSIS CATEGORY')
print('='*80)
print(f'\n{"Metric":<25}', end='')
for cat in categories:
    print(f'{cat[:12]:>14}', end='')
print()
print('-'*80)

metrics = ['N_total', 'National_rate', 'CV', 'Extremal_ratio', 'IQR_ratio', 'Mean_SDR']
labels = ['Total procedures', 'National rate/100k/yr', 'CV (%)', 'Extremal ratio', 'IQR ratio', 'Mean SDR']

for label, metric in zip(labels, metrics):
    print(f'{label:<25}', end='')
    for cat in categories:
        val = variation_results[cat][metric]
        if metric == 'N_total':
            print(f'{val:>14,}', end='')
        elif pd.isna(val):
            print(f'{"N/A":>14}', end='')
        else:
            print(f'{val:>14.2f}', end='')
    print()

In [ ]:
# ── 12c. Multi-panel Beeswarm Plot — Weinstein Figure Replication ──
# This is the signature figure: 4 panels side-by-side, SDR on log scale

fig, ax = plt.subplots(figsize=(12, 10))

# Prepare data for all categories
np.random.seed(42)
x_positions = {cat: i for i, cat in enumerate(categories)}

for cat in categories:
    cat_data = variation_results[cat]['state_data']
    x = x_positions[cat]
    sdrs = cat_data['SDR'].values
    
    # Jitter x-positions for beeswarm effect
    jitter = np.random.normal(0, 0.08, len(sdrs))
    
    ax.scatter(
        x + jitter, sdrs,
        s=50, c='#1f4e79', alpha=0.75, edgecolors='white', linewidths=0.4, zorder=3
    )

# Reference line at SDR = 1.0
ax.axhline(1.0, color='red', linestyle='-', linewidth=1.5, alpha=0.5, zorder=1)

# Log scale
ax.set_yscale('log')
ax.set_ylim(0.01, 100)
ax.set_yticks([0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0])
ax.get_yaxis().set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x:.0f}' if x >= 1 else f'{x:.2f}'
))
ax.yaxis.set_minor_formatter(mticker.NullFormatter())

# X-axis labels
ax.set_xticks(range(len(categories)))
ax.set_xticklabels([c.replace(' ', '\n') for c in categories], fontsize=11)
ax.set_xlim(-0.6, len(categories) - 0.4)

# Labels
ax.set_ylabel('Standardized Discharge Ratio (log scale)', fontsize=12)
ax.set_title('Geographic Variation by Diagnosis Category\n(Brazilian States, 2015–2020, à la Weinstein)', fontsize=13)

# Add variation metrics as a table below the plot
cell_text = []
row_labels = ['Mean rate/100k/yr', 'Extremal ratio', 'IQR ratio', 'CV (%)']
for metric_key in ['National_rate', 'Extremal_ratio', 'IQR_ratio', 'CV']:
    row = []
    for cat in categories:
        val = variation_results[cat][metric_key]
        if pd.isna(val):
            row.append('N/A')
        elif metric_key in ('Extremal_ratio', 'IQR_ratio'):
            row.append(f'{val:.2f}')
        elif metric_key == 'CV':
            row.append(f'{val:.1f}')
        else:
            row.append(f'{val:.2f}')
    cell_text.append(row)

table = ax.table(
    cellText=cell_text,
    rowLabels=row_labels,
    colLabels=[c.replace(' ', '\n') for c in categories],
    cellLoc='center',
    loc='bottom',
    bbox=[0.0, -0.32, 1.0, 0.22]
)
table.auto_set_font_size(False)
table.set_fontsize(9)
for key, cell in table.get_celld().items():
    if key[0] == 0:  # Header row
        cell.set_facecolor('#d4e6f1')
        cell.set_text_props(fontweight='bold')
    elif key[1] == -1:  # Row labels
        cell.set_facecolor('#f0f0f0')
        cell.set_text_props(fontweight='bold')

# Background
ax.set_facecolor('#fffff0')
ax.grid(axis='y', alpha=0.3)
ax.grid(axis='x', visible=False)

plt.subplots_adjust(bottom=0.28)
plt.savefig('output/weinstein_multipanel_beeswarm.png', dpi=200, bbox_inches='tight')
plt.show()

## 13. Municipality-Level Small Area Analysis

The Dartmouth Atlas methodology uses Hospital Referral Regions (HRRs) as the unit of analysis. Brazil's closest equivalent is the **município** (municipality). We analyze variation at this finer geographic granularity, filtering to municipalities with sufficient volume to produce stable rates (≥ 10 procedures over the study period).

In [ ]:
# ── 13a. Municipality-level variation ──

# Count procedures by patient residence municipality
muni_col = 'res_city_name' if 'res_city_name' in df.columns else 'int_MUNNOMEX'
state_col = 'res_SIGLA_UF'

muni_counts = df.groupby([state_col, muni_col]).size().reset_index(name='N')
muni_counts.columns = ['UF', 'Municipality', 'N']

print(f'Total municipalities with arthrodesis patients: {len(muni_counts)}')
print(f'Municipalities with ≥10 procedures: {(muni_counts["N"] >= 10).sum()}')
print(f'Municipalities with ≥50 procedures: {(muni_counts["N"] >= 50).sum()}')
print(f'Municipalities with ≥100 procedures: {(muni_counts["N"] >= 100).sum()}')

# Filter to municipalities with ≥10 procedures for stable rates
muni_stable = muni_counts[muni_counts['N'] >= 10].copy()
n_muni = len(muni_stable)

# For municipalities without population data, use procedure rate relative to state average
# Compute SDR relative to the state mean
state_means = muni_stable.groupby('UF')['N'].mean()
muni_stable['state_mean'] = muni_stable['UF'].map(state_means)
muni_stable['SDR_state'] = muni_stable['N'] / muni_stable['state_mean']

# Also compute SDR relative to national mean
national_muni_mean = muni_stable['N'].mean()
muni_stable['SDR_national'] = muni_stable['N'] / national_muni_mean

# Variation metrics at municipality level
rates_m = muni_stable['N']  # Using raw counts as proxy (no muni-level population)
sdrs_m = muni_stable['SDR_national']

cv_muni = rates_m.std() / rates_m.mean() * 100
extremal_muni = rates_m.max() / rates_m.min()
q1_m, q3_m = rates_m.quantile(0.25), rates_m.quantile(0.75)
iqr_ratio_muni = q3_m / q1_m

print(f'\n{"="*60}')
print(f'MUNICIPALITY-LEVEL VARIATION (N={n_muni} municipalities with ≥10 cases)')
print(f'{"="*60}')
print(f'  Mean procedures per municipality: {rates_m.mean():.1f}')
print(f'  Median:                           {rates_m.median():.0f}')
print(f'  Range:                            {rates_m.min():.0f} – {rates_m.max():.0f}')
print(f'  CV (%):                           {cv_muni:.1f}')
print(f'  Extremal ratio:                   {extremal_muni:.1f}')
print(f'  IQR ratio:                        {iqr_ratio_muni:.2f}')

# Top 15 municipalities
print(f'\nTOP 15 MUNICIPALITIES BY VOLUME:')
top15_muni = muni_stable.nlargest(15, 'N')
for _, row in top15_muni.iterrows():
    print(f'  {row["Municipality"]}, {row["UF"]}: {row["N"]:,} procedures')

In [ ]:
# ── 13b. Municipality-level beeswarm (log scale) ──

fig, ax = plt.subplots(figsize=(8, 10))

np.random.seed(42)
sdrs_sorted = muni_stable['SDR_national'].sort_values().values
jitter = np.random.normal(0, 0.12, len(sdrs_sorted))

ax.scatter(
    jitter, sdrs_sorted,
    s=15, c='#1f4e79', alpha=0.4, edgecolors='none', zorder=3
)

# Reference line
ax.axhline(1.0, color='red', linestyle='-', linewidth=1.5, alpha=0.6,
           label='National average (SDR=1.0)')

# IQR band
q1_sdr = np.percentile(sdrs_sorted, 25)
q3_sdr = np.percentile(sdrs_sorted, 75)
ax.axhspan(q1_sdr, q3_sdr, alpha=0.1, color='blue', label=f'IQR ({q1_sdr:.2f}–{q3_sdr:.2f})')

ax.set_yscale('log')
ax.set_ylim(0.01, 100)
ax.set_yticks([0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0])
ax.get_yaxis().set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x:.0f}' if x >= 1 else f'{x:.2f}'
))
ax.yaxis.set_minor_formatter(mticker.NullFormatter())

ax.set_xlim(-0.8, 0.8)
ax.set_xticks([0])
ax.set_xticklabels([f'Municipalities\n(N={n_muni})'])
ax.set_ylabel('Standardized Ratio (log scale)', fontsize=12)
ax.set_title(f'Geographic Variation at Municipality Level\n'
             f'(CV={cv_muni:.0f}%, Extremal={extremal_muni:.0f}, IQR ratio={iqr_ratio_muni:.1f})',
             fontsize=13)
ax.legend(loc='upper right', fontsize=9)

ax.set_facecolor('#fffff0')
ax.grid(axis='y', alpha=0.3)
ax.grid(axis='x', visible=False)

plt.tight_layout()
plt.savefig('output/weinstein_municipality_beeswarm.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 13c. Export variation metrics table ──

# Build exportable table
export_rows = []
# Overall
export_rows.append({
    'Category': 'All Arthrodesis',
    'N': len(df),
    'National_rate_100k_yr': national_rate,
    'CV_pct': cv,
    'Extremal_ratio': extremal_ratio,
    'IQR_ratio': iqr_ratio,
    'Q1_rate': q1,
    'Q3_rate': q3,
    'Mean_SDR': mean_sdr,
    'Level': 'State (N=27)'
})

# By category
for cat in categories:
    vr = variation_results[cat]
    export_rows.append({
        'Category': cat,
        'N': vr['N_total'],
        'National_rate_100k_yr': vr['National_rate'],
        'CV_pct': vr['CV'],
        'Extremal_ratio': vr['Extremal_ratio'],
        'IQR_ratio': vr['IQR_ratio'],
        'Q1_rate': vr['Q1'],
        'Q3_rate': vr['Q3'],
        'Mean_SDR': vr['Mean_SDR'],
        'Level': 'State (N=27)'
    })

# Municipality level (overall)
export_rows.append({
    'Category': 'All Arthrodesis (Municipality)',
    'N': muni_stable['N'].sum(),
    'National_rate_100k_yr': np.nan,  # No pop data
    'CV_pct': cv_muni,
    'Extremal_ratio': extremal_muni,
    'IQR_ratio': iqr_ratio_muni,
    'Q1_rate': q1_m,
    'Q3_rate': q3_m,
    'Mean_SDR': np.nan,
    'Level': f'Municipality (N={n_muni})'
})

variation_table = pd.DataFrame(export_rows)
variation_table.to_csv('output/table3_weinstein_variation_metrics.csv', index=False)

print('Table 3 — Weinstein Variation Metrics:')
print(variation_table.round(2).to_string(index=False))

# Final summary
print(f'\n{"="*60}')
print('WEINSTEIN ANALYSIS COMPLETE')
print(f'{"="*60}')
print(f'\nNew outputs saved:')
import os
for f in sorted(os.listdir('output')):
    if 'weinstein' in f or 'table3' in f:
        size = os.path.getsize(f'output/{f}')
        print(f'  {f} ({size/1024:.0f} KB)')

## 14. Enhanced Choropleth Maps

Multi-panel geographic visualizations: SDR maps, diagnosis-specific rate maps, bivariate rate–mortality choropleth, and Weinstein variation metric overlays.

In [ ]:
# ── 14a. SDR Choropleth — Diverging color scale centered at 1.0 ──
from matplotlib.colors import TwoSlopeNorm

states_sdr = states_gdf.merge(state_rates, how='left', left_on='abbrev_state', right_on='res_SIGLA_UF')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Panel 1: SDR with diverging colormap (blue=low, white=1.0, red=high)
norm = TwoSlopeNorm(vmin=states_sdr['SDR'].min(), vcenter=1.0, vmax=states_sdr['SDR'].max())
states_sdr.plot(
    column='SDR', cmap='RdBu_r', norm=norm, legend=True, ax=axes[0],
    edgecolor='gray', linewidth=0.5,
    legend_kwds={'label': 'SDR (1.0 = national avg)', 'orientation': 'horizontal', 'shrink': 0.7}
)
# Add state labels
for _, row in states_sdr.iterrows():
    centroid = row['geometry'].centroid
    axes[0].annotate(
        f"{row['abbrev_state']}\n{row['SDR']:.1f}",
        (centroid.x, centroid.y), fontsize=6, ha='center', va='center',
        fontweight='bold', color='black',
        bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.6, edgecolor='none')
    )
axes[0].set_title('Standardized Discharge Ratio by State', fontsize=13)
axes[0].axis('off')

# Panel 2: SDR category map
sdr_bins = [0, 0.4, 0.7, 1.0, 1.5, 99]
sdr_labels = ['Very low (<0.4)', 'Low (0.4–0.7)', 'Average (0.7–1.0)', 'Above avg (1.0–1.5)', 'High (>1.5)']
states_sdr['SDR_cat'] = pd.cut(states_sdr['SDR'], bins=sdr_bins, labels=sdr_labels)

cmap_cat = plt.cm.get_cmap('RdYlGn_r', 5)
states_sdr.plot(
    column='SDR_cat', cmap='RdYlGn_r', legend=True, ax=axes[1],
    edgecolor='gray', linewidth=0.5, categorical=True,
    legend_kwds={'title': 'SDR Category', 'loc': 'lower left', 'fontsize': 8}
)
axes[1].set_title('SDR Categories', fontsize=13)
axes[1].axis('off')

fig.suptitle('Geographic Variation — Standardized Discharge Ratio\n(Spinal Arthrodesis, SUS 2015–2020)', fontsize=14)
plt.tight_layout()
plt.savefig('output/choropleth_sdr.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 14b. Diagnosis-Specific Rate Maps (4-panel) ──

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    cat_state = variation_results[cat]['state_data']
    states_cat = states_gdf.merge(cat_state, how='left', left_on='abbrev_state', right_on='UF')
    states_cat['annual_rate'] = states_cat['annual_rate'].fillna(0)
    
    states_cat.plot(
        column='annual_rate', cmap='YlOrRd', legend=True, ax=axes[idx],
        edgecolor='gray', linewidth=0.5,
        legend_kwds={'label': 'Rate/100k/yr', 'orientation': 'horizontal',
                     'shrink': 0.6, 'pad': 0.02}
    )
    
    # State labels with rate values
    for _, row in states_cat.iterrows():
        centroid = row['geometry'].centroid
        rate_val = row['annual_rate']
        if rate_val > 0:
            axes[idx].annotate(
                f"{row['abbrev_state']}",
                (centroid.x, centroid.y), fontsize=5.5, ha='center', va='center',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.5, edgecolor='none')
            )
    
    vr = variation_results[cat]
    axes[idx].set_title(
        f'{cat}\n(N={vr["N_total"]:,}, CV={vr["CV"]:.0f}%, IQR ratio={vr["IQR_ratio"]:.1f})',
        fontsize=11
    )
    axes[idx].axis('off')

fig.suptitle('Diagnosis-Specific Arthrodesis Rates by State\n(Annual rate per 100,000, 2015–2020)', fontsize=14)
plt.tight_layout()
plt.savefig('output/choropleth_diagnosis_specific.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 14c. Bivariate Choropleth — Rate vs Mortality ──
# States colored by combination of high/low rate AND high/low mortality

# Merge rate and mortality data
bivar = states_gdf.merge(
    df_res[['UF', 'Rate_per_100k']], how='left', left_on='abbrev_state', right_on='UF'
)
bivar = bivar.merge(
    mortality_by_state[['res_SIGLA_UF', 'mortality_pct']], how='left',
    left_on='abbrev_state', right_on='res_SIGLA_UF'
)

# Classify into 3×3 grid
rate_terciles = pd.qcut(bivar['Rate_per_100k'], 3, labels=['Low rate', 'Mid rate', 'High rate'])
mort_terciles = pd.qcut(bivar['mortality_pct'], 3, labels=['Low mort', 'Mid mort', 'High mort'])
bivar['bivar_class'] = rate_terciles.astype(str) + ' / ' + mort_terciles.astype(str)

# Color mapping for 3x3 bivariate scheme
bivar_colors = {
    'Low rate / Low mort': '#e8e8e8',    # light gray — low everything
    'Low rate / Mid mort': '#b5c0da',    # light blue
    'Low rate / High mort': '#6c83b5',   # dark blue — low rate, high mort (worst access?)
    'Mid rate / Low mort': '#b8d6be',    # light green
    'Mid rate / Mid mort': '#90b2b3',    # teal
    'Mid rate / High mort': '#567994',   # dark teal
    'High rate / Low mort': '#73ae80',   # green — high rate, low mort (best)
    'High rate / Mid mort': '#5a9178',   # dark green
    'High rate / High mort': '#2a5a5b',  # very dark — high rate, high mort
}

bivar['color'] = bivar['bivar_class'].map(bivar_colors)
bivar['color'] = bivar['color'].fillna('#cccccc')

fig, ax = plt.subplots(figsize=(10, 10))
bivar.plot(color=bivar['color'], edgecolor='white', linewidth=0.8, ax=ax)

# State labels
for _, row in bivar.iterrows():
    centroid = row['geometry'].centroid
    ax.annotate(
        row['abbrev_state'],
        (centroid.x, centroid.y), fontsize=7, ha='center', va='center', fontweight='bold',
        color='white' if row['color'] in ['#6c83b5', '#567994', '#2a5a5b', '#5a9178'] else 'black'
    )

ax.set_title('Bivariate Map: Procedure Rate × Mortality\n(Terciles, SUS 2015–2020)', fontsize=13)
ax.axis('off')

# Build legend as an inset grid
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
ax_legend = inset_axes(ax, width='20%', height='20%', loc='lower left', borderpad=2)
legend_matrix = [
    ['#73ae80', '#5a9178', '#2a5a5b'],
    ['#b8d6be', '#90b2b3', '#567994'],
    ['#e8e8e8', '#b5c0da', '#6c83b5'],
]
for i, row_colors in enumerate(legend_matrix):
    for j, c in enumerate(row_colors):
        ax_legend.add_patch(plt.Rectangle((j, 2 - i), 1, 1, facecolor=c, edgecolor='white', linewidth=0.5))

ax_legend.set_xlim(0, 3)
ax_legend.set_ylim(0, 3)
ax_legend.set_xticks([0.5, 1.5, 2.5])
ax_legend.set_xticklabels(['Low', 'Mid', 'High'], fontsize=6)
ax_legend.set_yticks([0.5, 1.5, 2.5])
ax_legend.set_yticklabels(['Low', 'Mid', 'High'], fontsize=6)
ax_legend.set_xlabel('Rate →', fontsize=7)
ax_legend.set_ylabel('Mortality →', fontsize=7)
ax_legend.tick_params(length=0)

plt.tight_layout()
plt.savefig('output/choropleth_bivariate_rate_mortality.png', dpi=200, bbox_inches='tight')
plt.show()

## 15. Patient Flow & Referral Network Maps

Analyzing inter-state patient migration: comparing `res_SIGLA_UF` (patient residence) vs `int_SIGLA_UF` (hospital location) to identify:
- **Net importers** — states that attract patients from elsewhere (surgical hubs)
- **Net exporters** — states whose residents travel elsewhere for surgery (access barriers)
- **Flow corridors** — major origin–destination pairs
- **Micropoint density** — using `res_LATITUDE`/`res_LONGITUDE` for patient-level geographic resolution

In [ ]:
# ── 15a. Inter-state patient flow analysis ──

# Compare residence vs hospital state
df['same_state'] = df['res_SIGLA_UF'] == df['int_SIGLA_UF']
n_same = df['same_state'].sum()
n_diff = (~df['same_state']).sum()

print('='*60)
print('PATIENT FLOW ANALYSIS')
print('='*60)
print(f'  Same state (residence = hospital):   {n_same:,} ({n_same/len(df)*100:.1f}%)')
print(f'  Different state (cross-border):      {n_diff:,} ({n_diff/len(df)*100:.1f}%)')

# Build origin-destination flow matrix
flow = df[~df['same_state']].groupby(['res_SIGLA_UF', 'int_SIGLA_UF']).size().reset_index(name='N')
flow.columns = ['Origin', 'Destination', 'N']
flow = flow.sort_values('N', ascending=False)

print(f'\nTOP 20 INTER-STATE FLOW CORRIDORS:')
print(f'{"Origin":<8} {"→ Destination":<15} {"N":>8} {"% of cross-border":>18}')
print('-'*52)
for _, row in flow.head(20).iterrows():
    pct = row['N'] / n_diff * 100
    print(f'{row["Origin"]:<8} → {row["Destination"]:<12} {row["N"]:>8,} {pct:>17.1f}%')

# Net flow per state
exports = df[~df['same_state']].groupby('res_SIGLA_UF').size()   # patients leaving
imports = df[~df['same_state']].groupby('int_SIGLA_UF').size()   # patients arriving
residents = df.groupby('res_SIGLA_UF').size()                     # total residents
performed = df.groupby('int_SIGLA_UF').size()                     # total performed

net_flow = pd.DataFrame({
    'Residents': residents,
    'Performed': performed,
    'Exports': exports,
    'Imports': imports,
}).fillna(0).astype(int)
net_flow['Net_flow'] = net_flow['Imports'] - net_flow['Exports']
net_flow['Pct_imported'] = (net_flow['Imports'] / net_flow['Performed'] * 100).round(1)
net_flow['Pct_exported'] = (net_flow['Exports'] / net_flow['Residents'] * 100).round(1)
net_flow = net_flow.sort_values('Net_flow', ascending=False)

print(f'\nNET FLOW BY STATE (positive = net importer / surgical hub):')
print(net_flow.to_string())

In [ ]:
# ── 15b. Net Flow Choropleth Map ──

states_flow = states_gdf.merge(
    net_flow.reset_index().rename(columns={'index': 'UF'}),
    how='left', left_on='abbrev_state', right_on='UF'
)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Panel 1: Net flow (diverging — blue=exporter, red=importer)
norm_flow = TwoSlopeNorm(
    vmin=states_flow['Net_flow'].min(), vcenter=0,
    vmax=states_flow['Net_flow'].max()
)
states_flow.plot(
    column='Net_flow', cmap='RdBu', norm=norm_flow, legend=True, ax=axes[0],
    edgecolor='gray', linewidth=0.5,
    legend_kwds={'label': 'Net flow (+ = importer)', 'orientation': 'horizontal', 'shrink': 0.7}
)
for _, row in states_flow.iterrows():
    centroid = row['geometry'].centroid
    nf = row['Net_flow'] if not pd.isna(row['Net_flow']) else 0
    sign = '+' if nf > 0 else ''
    axes[0].annotate(
        f"{row['abbrev_state']}\n{sign}{int(nf)}",
        (centroid.x, centroid.y), fontsize=5.5, ha='center', va='center', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.6, edgecolor='none')
    )
axes[0].set_title('Net Patient Flow\n(Red = surgical hub, Blue = exporter)', fontsize=12)
axes[0].axis('off')

# Panel 2: % of procedures that are imported patients
states_flow.plot(
    column='Pct_imported', cmap='Purples', legend=True, ax=axes[1],
    edgecolor='gray', linewidth=0.5,
    legend_kwds={'label': '% imported patients', 'orientation': 'horizontal', 'shrink': 0.7}
)
for _, row in states_flow.iterrows():
    centroid = row['geometry'].centroid
    pct = row['Pct_imported'] if not pd.isna(row['Pct_imported']) else 0
    axes[1].annotate(
        f"{row['abbrev_state']}\n{pct:.0f}%",
        (centroid.x, centroid.y), fontsize=5.5, ha='center', va='center', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.6, edgecolor='none')
    )
axes[1].set_title('Percentage of Imported Patients\n(out of all procedures performed in state)', fontsize=12)
axes[1].axis('off')

fig.suptitle('Patient Migration for Spinal Arthrodesis (SUS 2015–2020)', fontsize=14)
plt.tight_layout()
plt.savefig('output/choropleth_patient_flow.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 15c. Flow Arrow Network Map — Top corridors on geographic base ──

# State centroids for arrow endpoints
state_centroids = {}
for _, row in states_gdf.iterrows():
    centroid = row['geometry'].centroid
    state_centroids[row['abbrev_state']] = (centroid.x, centroid.y)

# Top N flow corridors
top_flows = flow.head(30).copy()

fig, ax = plt.subplots(figsize=(12, 12))

# Draw base map
states_gdf.plot(ax=ax, color='#f0f0f0', edgecolor='gray', linewidth=0.5)

# Draw flow arrows
max_flow = top_flows['N'].max()
for _, frow in top_flows.iterrows():
    origin = frow['Origin']
    dest = frow['Destination']
    n = frow['N']
    
    if origin in state_centroids and dest in state_centroids:
        ox, oy = state_centroids[origin]
        dx, dy = state_centroids[dest]
        
        # Arrow width proportional to flow volume
        width = 0.3 + (n / max_flow) * 3.0
        alpha = 0.3 + (n / max_flow) * 0.5
        
        ax.annotate(
            '', xy=(dx, dy), xytext=(ox, oy),
            arrowprops=dict(
                arrowstyle='->', color='#c0392b', lw=width, alpha=alpha,
                connectionstyle='arc3,rad=0.15'
            )
        )

# State labels
for uf, (cx, cy) in state_centroids.items():
    ax.annotate(
        uf, (cx, cy), fontsize=6.5, ha='center', va='center', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.8, edgecolor='gray', linewidth=0.3)
    )

ax.set_title('Top 30 Inter-State Patient Flow Corridors\n(Arrow thickness ∝ patient volume)', fontsize=13)
ax.axis('off')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#c0392b', linewidth=3.5, alpha=0.8, label=f'Largest flow ({top_flows.iloc[0]["N"]:,})'),
    Line2D([0], [0], color='#c0392b', linewidth=1.5, alpha=0.5, label=f'Smallest shown ({top_flows.iloc[-1]["N"]:,})'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('output/network_flow_arrows.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 15d. Micropoint Density Map — Patient Residence Locations ──

# Check coordinate availability
lat_col = 'res_LATITUDE' if 'res_LATITUDE' in df.columns else None
lon_col = 'res_LONGITUDE' if 'res_LONGITUDE' in df.columns else None

if lat_col and lon_col:
    # Filter valid coordinates (within Brazil bounds)
    coords = df[[lat_col, lon_col]].dropna()
    coords = coords[
        (coords[lat_col] >= -34) & (coords[lat_col] <= 6) &
        (coords[lon_col] >= -74) & (coords[lon_col] <= -34)
    ]
    
    print(f'Valid geocoded records: {len(coords):,} / {len(df):,} ({len(coords)/len(df)*100:.1f}%)')
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 10))
    
    # Panel 1: Point scatter on state boundaries
    states_gdf.plot(ax=axes[0], color='#f5f5f5', edgecolor='gray', linewidth=0.5)
    axes[0].scatter(
        coords[lon_col], coords[lat_col],
        s=0.3, c='#c0392b', alpha=0.05, rasterized=True
    )
    axes[0].set_title(f'Patient Residence Locations\n(N={len(coords):,} geocoded procedures)', fontsize=12)
    axes[0].axis('off')
    
    # Panel 2: 2D histogram / heatmap
    states_gdf.plot(ax=axes[1], color='#f5f5f5', edgecolor='gray', linewidth=0.5)
    hb = axes[1].hexbin(
        coords[lon_col], coords[lat_col],
        gridsize=80, cmap='YlOrRd', mincnt=1, alpha=0.8,
        linewidths=0.1, edgecolors='none'
    )
    plt.colorbar(hb, ax=axes[1], shrink=0.5, label='Procedure count')
    axes[1].set_title('Procedure Density Heatmap\n(Hexagonal bins)', fontsize=12)
    axes[1].axis('off')
    
    fig.suptitle('Microgeographic Distribution of Arthrodesis Patients (SUS 2015–2020)', fontsize=14)
    plt.tight_layout()
    plt.savefig('output/micropoint_density_map.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # ── Cross-border patient scatter ──
    # Highlight patients who traveled to different states
    df_cross = df[~df['same_state']].copy()
    coords_cross = df_cross[[lat_col, lon_col]].dropna()
    coords_cross = coords_cross[
        (coords_cross[lat_col] >= -34) & (coords_cross[lat_col] <= 6) &
        (coords_cross[lon_col] >= -74) & (coords_cross[lon_col] <= -34)
    ]
    
    fig, ax = plt.subplots(figsize=(10, 10))
    states_gdf.plot(ax=ax, color='#f5f5f5', edgecolor='gray', linewidth=0.5)
    
    # All patients in gray
    ax.scatter(coords[lon_col], coords[lat_col],
               s=0.2, c='lightgray', alpha=0.03, rasterized=True, label='Same-state')
    # Cross-border in red
    ax.scatter(coords_cross[lon_col], coords_cross[lat_col],
               s=0.8, c='#c0392b', alpha=0.08, rasterized=True, label='Cross-border')
    
    ax.set_title(f'Cross-Border Patients (Red) vs Same-State (Gray)\n'
                 f'(N={len(coords_cross):,} cross-border of {len(coords):,} total)', fontsize=12)
    ax.legend(loc='lower right', fontsize=9, markerscale=20)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('output/micropoint_crossborder.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('Coordinate columns not found — skipping micropoint maps.')

# --- Section 16: Statistical Testing ---
## 16. Statistical Testing

**Objective:** Formal hypothesis testing for geographic variation and temporal trends.

- **16a** — Kruskal-Wallis tests across regions (rate, mortality, LOS, cost) + Dunn's post-hoc
- **16b** — Spearman correlation matrix (rate vs mortality vs specialists vs cost)
- **16c** — Global Moran's I for spatial autocorrelation of SDR
- **16d** — LISA (Local Indicators of Spatial Association) cluster map
- **16e** — Mann-Kendall trend test for temporal trends
- **16f** — Chi-square: diagnosis distribution across regions
- **16g** — Export all statistical results

In [ ]:
# ============================================================
# 16a — Kruskal-Wallis Tests Across Regions + Dunn's Post-Hoc
# ============================================================
from scipy import stats
import scikit_posthocs as sp

# Build state-level data with region assignment
state_region = df.groupby('res_SIGLA_UF').agg(
    region=('res_region', 'first'),
    n_procedures=('res_SIGLA_UF', 'size'),
    mean_age=('Age', 'mean'),
    mortality_pct=('death', lambda x: x.mean() * 100),
    mean_los=('Length_of_stay', 'mean'),
    mean_cost_brl=('Adj_VAL_TOTAL', 'mean'),
).reset_index()

# Merge population data for rates
state_region = state_region.merge(
    pop_df[['UF', 'Population']], left_on='res_SIGLA_UF', right_on='UF', how='left'
)
state_region['rate_100k_yr'] = state_region['n_procedures'] / state_region['Population'] * 100_000 / 6

# Detect actual region names from data
actual_regions = sorted(state_region['region'].unique())
print(f"Regions in data: {actual_regions}")

# ---- Kruskal-Wallis for each outcome ----
kw_results = []
outcomes = {
    'rate_100k_yr': 'Annual rate/100k',
    'mortality_pct': 'Mortality (%)',
    'mean_los': 'Mean LOS (days)',
    'mean_cost_brl': 'Mean cost (BRL)',
    'mean_age': 'Mean age (years)',
}

for var, label in outcomes.items():
    groups = [g[var].values for _, g in state_region.groupby('region')]
    h_stat, p_val = stats.kruskal(*groups)
    # Effect size: eta-squared for Kruskal-Wallis = (H - k + 1) / (n - k)
    k = len(groups)
    n = len(state_region)
    eta_sq = (h_stat - k + 1) / (n - k)
    eta_sq = max(0, eta_sq)  # floor at 0
    kw_results.append({
        'Outcome': label,
        'H_statistic': round(h_stat, 2),
        'df': k - 1,
        'p_value': p_val,
        'eta_squared': round(eta_sq, 3),
        'Significant': 'Yes' if p_val < 0.05 else 'No'
    })

kw_df = pd.DataFrame(kw_results)
print("=" * 70)
print("KRUSKAL-WALLIS TESTS: Differences Across 5 Brazilian Regions")
print("=" * 70)
print(f"H0: No difference in outcome across regions")
print(f"N = {n} states, k = {k} regions\n")
print(kw_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# ---- Dunn's post-hoc for significant outcomes ----
print("\n" + "=" * 70)
print("DUNN'S POST-HOC TESTS (Bonferroni correction)")
print("=" * 70)

dunn_tables = {}
for var, label in outcomes.items():
    h_stat, p_val = stats.kruskal(*[g[var].values for _, g in state_region.groupby('region')])
    if p_val < 0.05:
        dunn = sp.posthoc_dunn(state_region, val_col=var, group_col='region', p_adjust='bonferroni')
        dunn_tables[label] = dunn
        print(f"\n--- {label} (p={p_val:.4f}) ---")
        # Show only significant pairs
        sig_pairs = []
        for i_idx in range(len(dunn)):
            for j_idx in range(i_idx+1, len(dunn.columns)):
                r1, r2 = dunn.index[i_idx], dunn.columns[j_idx]
                pval = dunn.iloc[i_idx, j_idx]
                if pval < 0.05:
                    sig_pairs.append(f"  {r1} vs {r2}: p={pval:.4f} *")
        if sig_pairs:
            print('\n'.join(sig_pairs))
        else:
            print("  No pairwise comparisons significant after Bonferroni correction")
    else:
        print(f"\n--- {label}: Kruskal-Wallis not significant (p={p_val:.4f}), skipping post-hoc ---")

# ---- Visual: boxplots by region ----
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribution of Outcomes by Region\n(Kruskal-Wallis test across 5 regions)', fontsize=14, fontweight='bold')

# Use actual region names from the data, in geographic order
regions_order = ['South', 'Southeast', 'Middle West', 'Northeast', 'North']
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

for idx, (var, label) in enumerate(outcomes.items()):
    ax = axes.flat[idx]
    
    data_by_region = [state_region[state_region['region'] == r][var].values for r in regions_order]
    bp = ax.boxplot(data_by_region, labels=regions_order, patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    # Add individual state points (jittered)
    for i, r in enumerate(regions_order):
        vals = state_region[state_region['region'] == r][var].values
        jitter = np.random.normal(0, 0.05, size=len(vals))
        ax.scatter([i+1+j for j in jitter], vals, color=colors[i], s=40, zorder=5, 
                   edgecolor='black', linewidth=0.5)
    
    p_val = kw_df[kw_df['Outcome'] == label]['p_value'].values[0]
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    ax.set_title(f'{label}\nKW H={kw_df[kw_df["Outcome"]==label]["H_statistic"].values[0]:.1f}, p={p_val:.4f} {sig}', fontsize=10)
    ax.tick_params(axis='x', rotation=30)

# Remove empty subplot
axes.flat[5].set_visible(False)
plt.tight_layout()
plt.savefig('output/kruskal_wallis_regions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: output/kruskal_wallis_regions.png")

In [ ]:
# ============================================================
# 16b — Spearman Correlation Matrix
# ============================================================

# Build correlation dataset at state level
corr_data = state_region[['res_SIGLA_UF', 'rate_100k_yr', 'mortality_pct', 'mean_los', 'mean_cost_brl', 'mean_age']].copy()

# Add specialist density if available
if 'specialists_per_100k' not in corr_data.columns:
    # Re-create specialist data from Section 8
    specialist_counts = {
        'SP': 2847, 'RJ': 1156, 'MG': 1098, 'RS': 742, 'PR': 689,
        'BA': 412, 'SC': 387, 'PE': 345, 'GO': 305, 'DF': 298,
        'CE': 270, 'PA': 158, 'ES': 156, 'MT': 126, 'MS': 115,
        'MA': 107, 'RN': 95, 'PB': 88, 'PI': 73, 'SE': 63,
        'AL': 62, 'AM': 61, 'TO': 36, 'RO': 35, 'AC': 15,
        'AP': 11, 'RR': 10
    }
    spec_df = pd.DataFrame(list(specialist_counts.items()), columns=['UF', 'specialists'])
    corr_data = corr_data.merge(spec_df, left_on='res_SIGLA_UF', right_on='UF', how='left')
    corr_data = corr_data.merge(pop_df[['UF', 'Population']], on='UF', how='left')
    corr_data['specialists_per_100k'] = corr_data['specialists'] / corr_data['Population'] * 100_000

# Select numeric columns for correlation
corr_vars = ['rate_100k_yr', 'mortality_pct', 'mean_los', 'mean_cost_brl', 'mean_age', 'specialists_per_100k']
corr_labels = ['Rate/100k/yr', 'Mortality (%)', 'Mean LOS', 'Mean Cost (BRL)', 'Mean Age', 'Specialists/100k']

corr_matrix = corr_data[corr_vars].corr(method='spearman')
p_matrix = pd.DataFrame(np.ones((len(corr_vars), len(corr_vars))), index=corr_vars, columns=corr_vars)

for i in range(len(corr_vars)):
    for j in range(i+1, len(corr_vars)):
        rho, p = stats.spearmanr(corr_data[corr_vars[i]].dropna(), corr_data[corr_vars[j]].dropna())
        p_matrix.iloc[i, j] = p
        p_matrix.iloc[j, i] = p

# Heatmap with significance annotations
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Create annotation strings with significance stars
annot = np.empty_like(corr_matrix, dtype=object)
for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        r = corr_matrix.iloc[i, j]
        p = p_matrix.iloc[i, j]
        stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        annot[i, j] = f'{r:.2f}{stars}'

import seaborn as sns
hm = sns.heatmap(corr_matrix, annot=annot, fmt='', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                 xticklabels=corr_labels, yticklabels=corr_labels, ax=ax,
                 square=True, linewidths=0.5, cbar_kws={'label': 'Spearman ρ'})
ax.set_title('Spearman Correlation Matrix — State-Level Variables\n(* p<0.05, ** p<0.01, *** p<0.001)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/spearman_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key correlations
print("=" * 60)
print("KEY SPEARMAN CORRELATIONS")
print("=" * 60)
pairs_of_interest = [
    ('rate_100k_yr', 'specialists_per_100k', 'Rate vs Specialists'),
    ('rate_100k_yr', 'mortality_pct', 'Rate vs Mortality'),
    ('rate_100k_yr', 'mean_cost_brl', 'Rate vs Cost'),
    ('specialists_per_100k', 'mortality_pct', 'Specialists vs Mortality'),
    ('mean_los', 'mortality_pct', 'LOS vs Mortality'),
]
for v1, v2, label in pairs_of_interest:
    rho, p = stats.spearmanr(corr_data[v1].dropna(), corr_data[v2].dropna())
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  {label:30s}: ρ = {rho:+.3f}, p = {p:.4f} {sig}")
print("\nSaved: output/spearman_correlation_matrix.png")

In [ ]:
# ============================================================
# 16c — Global Moran's I: Spatial Autocorrelation of SDR
# ============================================================
import libpysal
from esda.moran import Moran

# Load state geometries
states_geo = geobr.read_state(year=2020)
states_geo['abbrev_state'] = states_geo['abbrev_state'].str.strip()

# Merge SDR data
sdr_data = state_region[['res_SIGLA_UF', 'rate_100k_yr']].copy()
national_rate = df.shape[0] / pop_df['Population'].sum() * 100_000 / 6
sdr_data['SDR'] = sdr_data['rate_100k_yr'] / national_rate

states_sdr = states_geo.merge(sdr_data, left_on='abbrev_state', right_on='res_SIGLA_UF', how='left')
states_sdr = states_sdr.dropna(subset=['SDR'])

# Build spatial weights (Queen contiguity)
w = libpysal.weights.Queen.from_dataframe(states_sdr)
w.transform = 'r'  # Row-standardize

# Global Moran's I
y = states_sdr['SDR'].values
moran = Moran(y, w, permutations=9999)

print("=" * 60)
print("GLOBAL MORAN'S I — Spatial Autocorrelation of SDR")
print("=" * 60)
print(f"  Moran's I:        {moran.I:.4f}")
print(f"  Expected I:       {moran.EI:.4f}")
print(f"  Variance:         {moran.VI_norm:.6f}")
print(f"  Z-score:          {moran.z_norm:.4f}")
print(f"  p-value (norm):   {moran.p_norm:.4f}")
print(f"  p-value (perm):   {moran.p_sim:.4f} (9999 permutations)")
print(f"  Permutation mean: {np.mean(moran.sim):.4f}")
print()
if moran.p_sim < 0.05:
    if moran.I > 0:
        print("  INTERPRETATION: Significant POSITIVE spatial autocorrelation.")
        print("  → States with similar SDR values tend to cluster geographically.")
        print("  → High-rate states neighbor other high-rate states (and vice versa).")
    else:
        print("  INTERPRETATION: Significant NEGATIVE spatial autocorrelation.")
        print("  → States with dissimilar SDR values tend to neighbor each other.")
else:
    print("  INTERPRETATION: No significant spatial autocorrelation detected.")
    print("  → SDR values are not significantly clustered in space.")

# Moran scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Moran scatter plot
ax = axes[0]
lag_y = libpysal.weights.lag_spatial(w, y)
# Standardize
y_std = (y - y.mean()) / y.std()
lag_std = (lag_y - lag_y.mean()) / lag_y.std()

ax.scatter(y_std, lag_std, c='steelblue', s=50, edgecolor='black', linewidth=0.5, alpha=0.7)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

# Fit line
m, b = np.polyfit(y_std, lag_std, 1)
x_line = np.linspace(y_std.min(), y_std.max(), 100)
ax.plot(x_line, m*x_line + b, 'r-', linewidth=2, label=f'slope = {m:.3f} (Moran\'s I)')

# Label quadrants
ax.text(0.95, 0.95, 'HH', transform=ax.transAxes, fontsize=14, fontweight='bold', color='red', ha='right', va='top')
ax.text(0.05, 0.95, 'LH', transform=ax.transAxes, fontsize=14, fontweight='bold', color='orange', ha='left', va='top')
ax.text(0.05, 0.05, 'LL', transform=ax.transAxes, fontsize=14, fontweight='bold', color='blue', ha='left', va='bottom')
ax.text(0.95, 0.05, 'HL', transform=ax.transAxes, fontsize=14, fontweight='bold', color='purple', ha='right', va='bottom')

# Label states
for i, row in states_sdr.iterrows():
    idx = states_sdr.index.get_loc(i)
    ax.annotate(row['abbrev_state'], (y_std[idx], lag_std[idx]), fontsize=7, ha='center', va='bottom')

ax.set_xlabel('SDR (standardized)', fontsize=11)
ax.set_ylabel('Spatial Lag of SDR (standardized)', fontsize=11)
ax.set_title(f"Moran Scatter Plot\nI = {moran.I:.3f}, p = {moran.p_sim:.4f}", fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: Reference distribution
ax2 = axes[1]
ax2.hist(moran.sim, bins=50, color='lightblue', edgecolor='gray', density=True, label='Permutation distribution')
ax2.axvline(moran.I, color='red', linewidth=2, linestyle='--', label=f'Observed I = {moran.I:.3f}')
ax2.axvline(moran.EI, color='black', linewidth=1, linestyle=':', label=f'Expected I = {moran.EI:.3f}')
ax2.set_xlabel("Moran's I", fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title(f"Reference Distribution (9999 permutations)\np-value = {moran.p_sim:.4f}", fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('output/moran_global_sdr.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: output/moran_global_sdr.png")

In [ ]:
# ============================================================
# 16d — LISA: Local Indicators of Spatial Association
# ============================================================
from esda.moran import Moran_Local

# Local Moran's I
lisa = Moran_Local(y, w, permutations=9999)

# Classify quadrants (1=HH, 2=LH, 3=LL, 4=HL)
# Only keep significant clusters (p < 0.05)
sig = lisa.p_sim < 0.05
quadrant = lisa.q.copy()
quadrant[~sig] = 0  # 0 = not significant

states_sdr = states_sdr.copy()
states_sdr['lisa_q'] = quadrant
states_sdr['lisa_I'] = lisa.Is
states_sdr['lisa_p'] = lisa.p_sim

# LISA cluster labels
lisa_labels = {0: 'Not significant', 1: 'High-High', 2: 'Low-High', 3: 'Low-Low', 4: 'High-Low'}
lisa_colors = {0: '#d9d9d9', 1: '#d7191c', 2: '#abd9e9', 3: '#2c7bb6', 4: '#fdae61'}
states_sdr['lisa_label'] = states_sdr['lisa_q'].map(lisa_labels)

# Print LISA results
print("=" * 60)
print("LOCAL MORAN'S I (LISA) — Spatial Clusters and Outliers")
print("=" * 60)
print(f"\nSignificant clusters/outliers (p < 0.05):")
for q_val, q_label in sorted(lisa_labels.items()):
    if q_val == 0:
        continue
    states_in_q = states_sdr[states_sdr['lisa_q'] == q_val]['abbrev_state'].tolist()
    if states_in_q:
        print(f"  {q_label}: {', '.join(states_in_q)}")

not_sig = states_sdr[states_sdr['lisa_q'] == 0]['abbrev_state'].tolist()
print(f"  Not significant: {', '.join(not_sig)}")

# LISA cluster map
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Local Indicators of Spatial Association (LISA)\nSpinal Arthrodesis SDR, SUS 2015–2020', 
             fontsize=14, fontweight='bold')

# Left: LISA cluster map
ax = axes[0]
states_sdr.plot(ax=ax, color=states_sdr['lisa_q'].map(lisa_colors), edgecolor='black', linewidth=0.5)
for _, row in states_sdr.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(row['abbrev_state'], xy=(centroid.x, centroid.y), fontsize=7, ha='center', va='center',
                fontweight='bold' if row['lisa_q'] != 0 else 'normal')

# Custom legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=lisa_colors[q], edgecolor='black', label=lisa_labels[q]) 
                  for q in [1, 2, 3, 4, 0]]
ax.legend(handles=legend_patches, loc='lower left', fontsize=9, title='LISA Cluster')
ax.set_title('LISA Cluster Map', fontsize=12)
ax.set_axis_off()

# Right: Local Moran's I significance map
ax2 = axes[1]
states_sdr['neg_log_p'] = -np.log10(states_sdr['lisa_p'].clip(lower=1e-5))
states_sdr.plot(ax=ax2, column='neg_log_p', cmap='YlOrRd', edgecolor='black', linewidth=0.5, legend=True,
                legend_kwds={'label': '-log10(p-value)', 'shrink': 0.6})
for _, row in states_sdr.iterrows():
    centroid = row.geometry.centroid
    ax2.annotate(row['abbrev_state'], xy=(centroid.x, centroid.y), fontsize=7, ha='center', va='center')
ax2.set_title('LISA Significance Map\n(-log10 p-value; darker = more significant)', fontsize=12)
ax2.set_axis_off()

plt.tight_layout()
plt.savefig('output/lisa_cluster_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: output/lisa_cluster_map.png")

In [ ]:
# ============================================================
# 16e — Mann-Kendall Trend Test for Temporal Trends
# ============================================================
import pymannkendall as mk

# Yearly aggregates
yearly = df.groupby('Year').agg(
    n_procedures=('Year', 'size'),
    mortality_pct=('death', lambda x: x.mean() * 100),
    mean_los=('Length_of_stay', 'mean'),
    mean_cost=('Adj_VAL_TOTAL', 'mean'),
    mean_age=('Age', 'mean'),
    icu_pct=('UCI_use', lambda x: x.mean() * 100),
).reset_index()

# National population for rate calculation
total_pop = pop_df['Population'].sum()
yearly['rate_100k'] = yearly['n_procedures'] / total_pop * 100_000

print("=" * 60)
print("MANN-KENDALL TREND TESTS (2015–2020)")
print("=" * 60)
print("H0: No monotonic trend over time\n")

mk_results = []
trend_vars = {
    'rate_100k': 'Procedure rate/100k',
    'mortality_pct': 'Mortality (%)',
    'mean_los': 'Mean LOS (days)',
    'mean_cost': 'Mean cost (BRL)',
    'mean_age': 'Mean age (years)',
    'icu_pct': 'ICU use (%)',
}

for var, label in trend_vars.items():
    result = mk.original_test(yearly[var].values)
    mk_results.append({
        'Variable': label,
        'Trend': result.trend,
        'Tau': round(result.Tau, 3),
        'S': result.s,
        'p_value': round(result.p, 4),
        'Slope_Sen': round(result.slope, 4),
        'Significant': 'Yes' if result.p < 0.05 else 'No'
    })
    direction = '↑' if result.slope > 0 else '↓'
    sig = '***' if result.p < 0.001 else '**' if result.p < 0.01 else '*' if result.p < 0.05 else 'ns'
    print(f"  {label:25s}: τ={result.Tau:+.3f}, Sen slope={result.slope:+.4f}, p={result.p:.4f} {sig} {direction}")

mk_df = pd.DataFrame(mk_results)

print("\n" + "-" * 60)
print("NOTE: 2020 was heavily affected by COVID-19 pandemic.")
print("Trends should be interpreted with caution as the 6-year")
print("window includes a major disruption in the final year.")

# ---- Trend visualization with Sen's slopes ----
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Temporal Trends 2015–2020 with Mann-Kendall Test\n(Sen's slope shown as red dashed line)", 
             fontsize=14, fontweight='bold')

for idx, (var, label) in enumerate(trend_vars.items()):
    ax = axes.flat[idx]
    years = yearly['Year'].values
    vals = yearly[var].values
    
    ax.plot(years, vals, 'o-', color='steelblue', linewidth=2, markersize=8)
    
    # Sen's slope line
    result = mk.original_test(vals)
    intercept = np.median(vals) - result.slope * np.median(years)
    trend_line = result.slope * years + intercept
    ax.plot(years, trend_line, 'r--', linewidth=1.5, alpha=0.7, label=f"Sen's slope = {result.slope:+.3f}")
    
    # COVID marker
    ax.axvspan(2019.5, 2020.5, alpha=0.15, color='red')
    
    sig = '***' if result.p < 0.001 else '**' if result.p < 0.01 else '*' if result.p < 0.05 else 'ns'
    ax.set_title(f'{label}\nτ={result.Tau:+.3f}, p={result.p:.4f} {sig}', fontsize=10)
    ax.set_xlabel('Year')
    ax.legend(fontsize=8)
    ax.set_xticks(years)

plt.tight_layout()
plt.savefig('output/mann_kendall_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: output/mann_kendall_trends.png")

In [ ]:
# ============================================================
# 16f — Chi-Square: Diagnosis Distribution Across Regions
# ============================================================

# Diagnosis categories (from Section 12)
def classify_diagnosis(icd):
    if pd.isna(icd):
        return 'Other'
    icd = str(icd).strip().upper()
    if any(icd.startswith(p) for p in ['M51', 'M47', 'M48.0']):
        if icd.startswith('M47.2'):
            return 'Cervical degenerative'
        return 'Lumbar degenerative'
    elif icd.startswith('M50') or icd.startswith('M47.2'):
        return 'Cervical degenerative'
    elif any(icd.startswith(p) for p in ['S12', 'S22', 'S32', 'S13', 'S14', 'S24', 'S34', 'T91']):
        return 'Trauma'
    elif any(icd.startswith(p) for p in ['M40', 'M41', 'M43.1']):
        return 'Deformity'
    else:
        return 'Other'

df['diag_category'] = df['Main_diagnosis'].apply(classify_diagnosis)

# Contingency table: Region × Diagnosis
ct = pd.crosstab(df['res_region'], df['diag_category'])
ct_display = ct[['Lumbar degenerative', 'Cervical degenerative', 'Trauma', 'Deformity', 'Other']]

print("=" * 70)
print("CONTINGENCY TABLE: Region × Diagnosis Category")
print("=" * 70)
print(ct_display)
print()

# Chi-square test
chi2, p_chi, dof, expected = stats.chi2_contingency(ct_display)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"Degrees of freedom:   {dof}")
print(f"p-value:              {p_chi:.2e}")
print(f"Cramér's V:           {np.sqrt(chi2 / (ct_display.values.sum() * (min(ct_display.shape) - 1))):.3f}")

if p_chi < 0.05:
    print("\nINTERPRETATION: Diagnosis distribution is SIGNIFICANTLY different across regions.")
else:
    print("\nINTERPRETATION: No significant difference in diagnosis distribution across regions.")

# ---- Standardized residuals ----
# Positive = over-represented, Negative = under-represented
std_resid = (ct_display.values - expected) / np.sqrt(expected)
std_resid_df = pd.DataFrame(std_resid, index=ct_display.index, columns=ct_display.columns)

print("\n" + "=" * 70)
print("STANDARDIZED RESIDUALS (>|1.96| = significant at α=0.05)")
print("=" * 70)
print(std_resid_df.round(2))

# ---- Visualization: stacked proportions + residual heatmap ----
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Diagnosis Distribution Across Regions\n(Chi-square test of independence)', 
             fontsize=14, fontweight='bold')

# Left: stacked bar (proportions)
ax = axes[0]
ct_pct = ct_display.div(ct_display.sum(axis=1), axis=0) * 100
colors = ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#d9d9d9']
ct_pct.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title(f'Diagnosis Proportions by Region\nχ²={chi2:.1f}, df={dof}, p={p_chi:.2e}', fontsize=11)
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('')
ax.legend(title='Diagnosis', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=0)
ax.set_ylim(0, 100)

# Right: standardized residuals heatmap
ax2 = axes[1]
sns.heatmap(std_resid_df, annot=True, fmt='.1f', cmap='RdBu_r', center=0, 
            linewidths=0.5, ax=ax2, cbar_kws={'label': 'Std. Residual'})
ax2.set_title('Standardized Residuals\n(Red = over-represented, Blue = under-represented)', fontsize=11)
ax2.set_ylabel('')

plt.tight_layout()
plt.savefig('output/chisquare_diagnosis_region.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: output/chisquare_diagnosis_region.png")

In [ ]:
# ============================================================
# 16g — Export All Statistical Results
# ============================================================

# Table 9: Kruskal-Wallis results
kw_df.to_csv('output/table9_kruskal_wallis.csv', index=False)
print("Saved: output/table9_kruskal_wallis.csv")

# Table 10: Mann-Kendall trend results
mk_df.to_csv('output/table10_mann_kendall_trends.csv', index=False)
print("Saved: output/table10_mann_kendall_trends.csv")

# Table 11: Spearman correlations (full matrix)
corr_out = corr_matrix.copy()
corr_out.index = corr_labels
corr_out.columns = corr_labels
corr_out.to_csv('output/table11_spearman_correlations.csv')
print("Saved: output/table11_spearman_correlations.csv")

# Table 12: LISA results by state
lisa_export = states_sdr[['abbrev_state', 'SDR', 'lisa_I', 'lisa_p', 'lisa_label']].copy()
lisa_export.columns = ['State', 'SDR', 'Local_Morans_I', 'p_value', 'LISA_cluster']
lisa_export = lisa_export.sort_values('SDR', ascending=False)
lisa_export.to_csv('output/table12_lisa_results.csv', index=False)
print("Saved: output/table12_lisa_results.csv")

# Table 13: Chi-square contingency + residuals
ct_display.to_csv('output/table13_chisquare_contingency.csv')
std_resid_df.to_csv('output/table13b_standardized_residuals.csv')
print("Saved: output/table13_chisquare_contingency.csv")
print("Saved: output/table13b_standardized_residuals.csv")

# Summary
print("\n" + "=" * 60)
print("SECTION 16 — STATISTICAL TESTING SUMMARY")
print("=" * 60)
print(f"\n  Kruskal-Wallis:     {sum(kw_df['Significant']=='Yes')}/{len(kw_df)} outcomes significant across regions")
print(f"  Moran's I:          {moran.I:.3f} (p={moran.p_sim:.4f}) — {'Significant' if moran.p_sim < 0.05 else 'Not significant'}")
print(f"  LISA clusters:      {sum(states_sdr['lisa_q'] != 0)} states in significant clusters")
print(f"  Mann-Kendall:       {sum(mk_df['Significant']=='Yes')}/{len(mk_df)} trends significant")
print(f"  Chi-square:         χ²={chi2:.1f}, p={p_chi:.2e} — {'Significant' if p_chi < 0.05 else 'Not significant'}")
print(f"\n  New figures: 5 (kruskal_wallis, spearman, moran, lisa, mann_kendall, chisquare)")
print(f"  New tables: 5 (tables 9-13)")
print(f"\nTotal output files: {len([f for f in os.listdir('output') if not f.startswith('.')])}")

# --- Section 17: Sankey & Chord Diagrams ---
## 17. Patient Flow Sankey & Chord Diagrams

**Objective:** Visualize inter-state patient flow using:
- **17a** — Sankey diagram (origin region → destination state) for top corridors
- **17b** — Chord diagram showing bidirectional flow between states

In [ ]:
# ============================================================
# 17a — Sankey Diagram: Origin State → Destination State
# ============================================================
from matplotlib.sankey import Sankey

# Get cross-state flows
df['same_state'] = df['res_SIGLA_UF'] == df['int_SIGLA_UF']
cross = df[~df['same_state']].copy()

# Build flow data: top corridors grouped by origin region → destination state
flow_data = cross.groupby(['res_SIGLA_UF', 'int_SIGLA_UF']).size().reset_index(name='N')
flow_data = flow_data.sort_values('N', ascending=False)

# For Sankey: use origin states (left) → destination states (right)
# Take top 20 corridors
top_flows = flow_data.head(20).copy()

# Build node lists
origins = top_flows['res_SIGLA_UF'].unique().tolist()
destinations = top_flows['int_SIGLA_UF'].unique().tolist()
# Remove overlaps (states that appear on both sides)
all_nodes = []
origin_labels = []
dest_labels = []
for s in origins:
    label = f'{s} (origin)'
    origin_labels.append(label)
    all_nodes.append(label)
for s in destinations:
    label = f'{s} (dest)'
    dest_labels.append(label)
    all_nodes.append(label)

node_idx = {n: i for i, n in enumerate(all_nodes)}

# Build links
sources = []
targets = []
values = []
for _, row in top_flows.iterrows():
    src = f"{row['res_SIGLA_UF']} (origin)"
    tgt = f"{row['int_SIGLA_UF']} (dest)"
    sources.append(node_idx[src])
    targets.append(node_idx[tgt])
    values.append(row['N'])

# Assign colors by region
region_map = df.drop_duplicates('res_SIGLA_UF').set_index('res_SIGLA_UF')['res_region'].to_dict()
region_colors = {
    'South': '#e41a1c', 'Southeast': '#377eb8', 'Middle West': '#4daf4a',
    'Northeast': '#984ea3', 'North': '#ff7f00'
}

node_colors = []
for label in all_nodes:
    state = label.split(' ')[0]
    region = region_map.get(state, 'Other')
    color = region_colors.get(region, '#999999')
    node_colors.append(color)

link_colors = []
for s in sources:
    c = node_colors[s]
    # Make semi-transparent
    link_colors.append(c + '40')  # hex alpha

# Build matplotlib-based Sankey using alluvial-style
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(-0.5, 2.5)

# Position nodes: origins on left (x=0), destinations on right (x=2)
# Stack vertically proportional to flow
origin_totals = top_flows.groupby('res_SIGLA_UF')['N'].sum().sort_values(ascending=False)
dest_totals = top_flows.groupby('int_SIGLA_UF')['N'].sum().sort_values(ascending=False)

total_flow = top_flows['N'].sum()
gap = 0.02  # gap between nodes as fraction of total

# Calculate y positions for origins
y_pos_origin = {}
y_current = 0
for state, total in origin_totals.items():
    height = total / total_flow
    y_pos_origin[state] = (y_current, y_current + height)
    y_current += height + gap

# Normalize so total height = 1
scale = 1 / y_current
y_pos_origin = {s: (b*scale, t*scale) for s, (b, t) in y_pos_origin.items()}

# Calculate y positions for destinations
y_pos_dest = {}
y_current = 0
for state, total in dest_totals.items():
    height = total / total_flow
    y_pos_dest[state] = (y_current, y_current + height)
    y_current += height + gap

scale = 1 / y_current
y_pos_dest = {s: (b*scale, t*scale) for s, (b, t) in y_pos_dest.items()}

# Draw origin rectangles
bar_width = 0.15
for state, (bottom, top) in y_pos_origin.items():
    region = region_map.get(state, 'Other')
    color = region_colors.get(region, '#999999')
    ax.barh(y=(bottom+top)/2, width=bar_width, height=top-bottom, left=-bar_width/2, 
            color=color, edgecolor='white', linewidth=0.5)
    ax.text(-bar_width/2 - 0.02, (bottom+top)/2, state, ha='right', va='center', fontsize=9, fontweight='bold')

# Draw destination rectangles
for state, (bottom, top) in y_pos_dest.items():
    region = region_map.get(state, 'Other')
    color = region_colors.get(region, '#999999')
    ax.barh(y=(bottom+top)/2, width=bar_width, height=top-bottom, left=2-bar_width/2,
            color=color, edgecolor='white', linewidth=0.5)
    ax.text(2+bar_width/2 + 0.02, (bottom+top)/2, state, ha='left', va='center', fontsize=9, fontweight='bold')

# Draw flows as bezier curves
from matplotlib.patches import FancyArrowPatch
from matplotlib.path import Path
import matplotlib.patches as mpatches

# Track consumed height for stacking flows within each node
origin_consumed = {s: y_pos_origin[s][0] for s in y_pos_origin}
dest_consumed = {s: y_pos_dest[s][0] for s in y_pos_dest}

for _, row in top_flows.iterrows():
    src = row['res_SIGLA_UF']
    dst = row['int_SIGLA_UF']
    n = row['N']
    
    # Flow height proportional to patient count
    src_total = origin_totals[src]
    dst_total = dest_totals[dst]
    src_height_frac = n / total_flow
    src_height = src_height_frac * (y_pos_origin[src][1] - y_pos_origin[src][0]) / (src_total / total_flow)
    dst_height = src_height_frac * (y_pos_dest[dst][1] - y_pos_dest[dst][0]) / (dst_total / total_flow)
    
    # Source y range
    y_src_bottom = origin_consumed[src]
    y_src_top = y_src_bottom + src_height
    origin_consumed[src] = y_src_top
    
    # Destination y range
    y_dst_bottom = dest_consumed[dst]
    y_dst_top = y_dst_bottom + dst_height
    dest_consumed[dst] = y_dst_top
    
    # Draw filled bezier band
    x_src = bar_width / 2
    x_dst = 2 - bar_width / 2
    x_mid = 1.0
    
    region = region_map.get(src, 'Other')
    color = region_colors.get(region, '#999999')
    
    # Upper path: src_top → dst_top
    verts_upper = [(x_src, y_src_top), (x_mid, y_src_top), (x_mid, y_dst_top), (x_dst, y_dst_top)]
    # Lower path: dst_bottom → src_bottom (reversed)
    verts_lower = [(x_dst, y_dst_bottom), (x_mid, y_dst_bottom), (x_mid, y_src_bottom), (x_src, y_src_bottom)]
    
    codes_upper = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
    codes_lower = [Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
    
    verts = verts_upper + verts_lower + [(x_src, y_src_top)]
    codes = codes_upper + codes_lower + [Path.CLOSEPOLY]
    
    path = Path(verts, codes)
    patch = mpatches.PathPatch(path, facecolor=color, alpha=0.35, edgecolor=color, linewidth=0.3)
    ax.add_patch(patch)
    
    # Label flows > 40 patients
    if n >= 40:
        ax.text(1.0, (y_src_top + y_src_bottom + y_dst_top + y_dst_bottom) / 4, 
                str(n), ha='center', va='center', fontsize=7, color='black', fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=r) for r, c in region_colors.items()]
ax.legend(handles=legend_patches, loc='upper right', title='Region', fontsize=9)

ax.set_title('Sankey Diagram — Inter-State Patient Flow\n(Top 20 corridors, SUS 2015–2020)', 
             fontsize=14, fontweight='bold')
ax.text(0, -0.05, 'Origin\n(patient residence)', ha='center', fontsize=11, fontstyle='italic')
ax.text(2, -0.05, 'Destination\n(hospital state)', ha='center', fontsize=11, fontstyle='italic')
ax.set_ylim(-0.08, 1.05)
ax.axis('off')

plt.tight_layout()
plt.savefig('output/sankey_patient_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: output/sankey_patient_flow.png")

In [ ]:
# ============================================================
# 17b — Chord Diagram: Bidirectional Inter-State Flow
# ============================================================

# Build symmetric flow matrix for top states involved in cross-state flow
# Select states with >= 20 total cross-state patients (in + out)
state_involvement = pd.concat([
    cross['res_SIGLA_UF'].value_counts().rename('out'),
    cross['int_SIGLA_UF'].value_counts().rename('into')
], axis=1).fillna(0)
state_involvement['total'] = state_involvement['out'] + state_involvement['into']
active_states = state_involvement[state_involvement['total'] >= 20].index.tolist()

# Build flow matrix among active states
flow_matrix = cross[cross['res_SIGLA_UF'].isin(active_states) & 
                     cross['int_SIGLA_UF'].isin(active_states)]\
    .groupby(['res_SIGLA_UF', 'int_SIGLA_UF']).size().unstack(fill_value=0)

# Reindex to ensure all active states present
flow_matrix = flow_matrix.reindex(index=active_states, columns=active_states, fill_value=0)
n_states = len(active_states)

print(f"Chord diagram: {n_states} states with >=20 cross-state patients")
print(f"Flow matrix shape: {flow_matrix.shape}")

# === Build chord diagram with matplotlib ===
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={'projection': 'polar'})

# State arc sizes proportional to total flow (in + out)
state_totals = flow_matrix.sum(axis=0) + flow_matrix.sum(axis=1)
total_all = state_totals.sum()

# Assign angular positions
gap_angle = 2 * np.pi * 0.02  # small gap between arcs
available = 2 * np.pi - n_states * gap_angle
arc_angles = {}
theta_start = 0

for state in active_states:
    arc_len = available * state_totals[state] / total_all
    arc_angles[state] = (theta_start, theta_start + arc_len)
    theta_start += arc_len + gap_angle

# Draw outer arcs (state segments)
for state, (start, end) in arc_angles.items():
    region = region_map.get(state, 'Other')
    color = region_colors.get(region, '#999999')
    
    # Draw arc
    theta = np.linspace(start, end, 100)
    r_outer = 1.0
    r_inner = 0.95
    ax.fill_between(theta, r_inner, r_outer, color=color, alpha=0.9, linewidth=0)
    
    # Label
    mid_theta = (start + end) / 2
    ax.text(mid_theta, 1.12, state, ha='center', va='center', fontsize=8, fontweight='bold',
            rotation=np.degrees(mid_theta) - 90 if mid_theta < np.pi else np.degrees(mid_theta) + 90,
            rotation_mode='anchor')

# Draw chords (flow connections)
# Track position within each arc for stacking
arc_consumed = {s: arc_angles[s][0] for s in active_states}

for src in active_states:
    for dst in active_states:
        if src == dst:
            continue
        flow_val = flow_matrix.loc[src, dst] if src in flow_matrix.index and dst in flow_matrix.columns else 0
        if flow_val < 5:  # skip very small flows
            continue
        
        # Source arc position
        src_arc_len = (arc_angles[src][1] - arc_angles[src][0]) * flow_val / state_totals[src]
        src_start = arc_consumed[src]
        src_end = src_start + src_arc_len
        arc_consumed[src] = src_end
        
        # Destination arc position  
        dst_arc_len = (arc_angles[dst][1] - arc_angles[dst][0]) * flow_val / state_totals[dst]
        dst_start = arc_consumed[dst]
        dst_end = dst_start + dst_arc_len
        arc_consumed[dst] = dst_end
        
        # Draw bezier chord
        src_mid = (src_start + src_end) / 2
        dst_mid = (dst_start + dst_end) / 2
        
        region = region_map.get(src, 'Other')
        color = region_colors.get(region, '#999999')
        
        # Bezier through center
        n_pts = 50
        t = np.linspace(0, 1, n_pts)
        # Quadratic bezier: P0=src, P1=center(0,0), P2=dst
        r0 = 0.93
        theta_pts = (1-t)**2 * src_mid + 2*(1-t)*t * ((src_mid+dst_mid)/2) + t**2 * dst_mid
        r_pts = (1-t)**2 * r0 + 2*(1-t)*t * 0.0 + t**2 * r0  # dips to center
        
        # Scale linewidth by flow
        lw = max(0.5, flow_val / 30)
        alpha = min(0.6, 0.15 + flow_val / 200)
        
        ax.plot(theta_pts, r_pts, color=color, alpha=alpha, linewidth=lw)

# Styling
ax.set_ylim(0, 1.25)
ax.set_yticks([])
ax.set_xticks([])
ax.spines['polar'].set_visible(False)
ax.grid(False)

# Legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=r) for r, c in region_colors.items()]
ax.legend(handles=legend_patches, loc='center', fontsize=9, title='Region',
          bbox_to_anchor=(0.5, 0.5), framealpha=0.8)

ax.set_title('Chord Diagram — Inter-State Patient Flow\n(Spinal Arthrodesis, SUS 2015–2020)',
             fontsize=14, fontweight='bold', pad=30)

plt.tight_layout()
plt.savefig('output/chord_patient_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: output/chord_patient_flow.png")
print(f"\nTotal cross-state patients shown: {flow_matrix.values.sum()}")
print(f"States in diagram: {', '.join(active_states)}")